# Chapter 2: Cryptography

> "Cryptography is the essential building block of independence for organizations on the Internet,
> just like armies are the essential building blocks of states." -- Julian Assange, paraphrasing a
> common sentiment in the cypherpunk movement

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Define the goals of cryptography (confidentiality, integrity, authentication, non-repudiation) and
   state Kerckhoffs's principle.
2. Break classical ciphers using frequency analysis and explain why they fail.
3. Explain perfect secrecy and the one-time pad, and why perfect secrecy is impractical at scale.
4. Distinguish true randomness, pseudo-randomness, and cryptographically secure pseudo-randomness, and
   identify the danger of weak generators.
5. Describe symmetric encryption, contrast stream and block ciphers, and explain the AES structure.
6. Compare the block-cipher modes of operation (ECB, CBC, CFB, OFB, CTR, GCM) and justify why ECB is
   insecure.
7. Explain cryptographic hash functions, their security properties, and constructions such as
   Merkle-Damgard and Merkle trees.
8. Explain message authentication codes (HMAC), authenticated encryption (AEAD), key derivation
   functions, and secure password storage.
9. Describe public-key cryptography: RSA, Diffie-Hellman key exchange, and elliptic-curve cryptography.
10. Explain digital signatures, public-key infrastructure, certificates, and the TLS handshake.
11. Describe advanced and forward-looking topics: homomorphic encryption, functional encryption,
    oblivious computation, steganography, and post-quantum cryptography.

## Key Terms

- **Plaintext / ciphertext**: the readable message and its encrypted form.
- **Cipher**: an algorithm for encryption and decryption.
- **Key**: the secret parameter that controls a cipher's output.
- **Symmetric cryptography**: encryption and decryption use the same shared secret key.
- **Asymmetric (public-key) cryptography**: a mathematically linked key pair, public and private.
- **Kerckhoffs's principle**: a cryptosystem must be secure even if everything except the key is public.
- **Cryptanalysis**: the study of breaking cryptographic systems.
- **Hash function**: a one-way function mapping arbitrary input to a fixed-size digest.
- **MAC**: message authentication code, a keyed tag proving integrity and authenticity.
- **AEAD**: authenticated encryption with associated data; confidentiality plus integrity in one step.
- **KDF**: key derivation function, deriving keys from passwords or other key material.
- **PKI**: public-key infrastructure, the system of certificate authorities and certificates that binds
  public keys to identities.


```{note}
Imagine passing a note in class. Anyone who intercepts it can read it, unless you and your friend
agreed in advance on a secret way to scramble the letters. Cryptography is the mathematics of doing
this so well that even someone with a supercomputer and the full description of your scrambling method,
but not your secret key, cannot read the note. The rest of this chapter turns that intuition into
precise, testable guarantees.
```

## 2.1 What Cryptography Is and What It Promises

Cryptography is the science of securing communication and data in the presence of adversaries. It is
the mathematical engine beneath nearly every security control in this book: it is how a website proves
its identity, how a password is stored safely, how a software update is shown to be authentic, and how
a messaging app keeps conversations private. The word comes from the Greek for "hidden writing," but
modern cryptography reaches far beyond secrecy. It provides four distinct services, and keeping them
separate in your mind is essential.

**Confidentiality** ensures that only authorized parties can read a message; this is encryption in the
everyday sense. **Integrity** ensures that a message has not been altered, so the recipient can detect
tampering. **Authentication** ensures that a message genuinely comes from its claimed sender.
**Non-repudiation** ensures that a sender cannot later deny having sent a message, a stronger property
that requires public-key signatures. A single protocol often combines several of these; for example,
when you log in to a bank, TLS provides confidentiality and integrity for the connection and
authenticates the server to you.

A foundational rule governs all serious cryptography: **Kerckhoffs's principle**, which states that a
cryptosystem should remain secure even if everything about the system, except the secret key, is public
knowledge. The modern restatement, attributed to Claude Shannon, is to assume "the enemy knows the
system." This is why reputable cryptographic algorithms are published, standardized, and subjected to
years of public analysis rather than kept secret. Security that depends on hiding the algorithm,
disparagingly called *security through obscurity*, fails the moment the algorithm leaks or is reverse
engineered, and history is littered with such failures. When you design or select cryptography, trust
only algorithms that have survived sustained public scrutiny, and never invent your own cipher for
production use.

It is equally important to know cryptography's limits. Encryption protects data, not the endpoints:
if an attacker controls your device, the strongest cipher will not save you, because the plaintext is
right there. Cryptography also cannot fix a bad protocol around it, cannot compensate for a weak or
leaked key, and is frequently undone not by breaking the mathematics but by implementation mistakes:
predictable randomness, reused keys, timing side channels, and downgrade attacks. Throughout this
chapter, watch how often the *system* fails even when the *algorithm* is sound.

### Encoding versus Encryption versus Hashing

Three operations are constantly confused, even by practitioners, yet they serve entirely different
purposes, and conflating them causes real security failures (for example, treating Base64 *encoding* as
if it protected data). The distinction turns on two questions: is the transformation reversible, and does
it depend on a secret key?

*Encoding* transforms data into another format using a public, keyless, reversible scheme, purely for
compatibility or transport, not secrecy. ASCII, Unicode, Base64, and URL-encoding are encodings: anyone
who knows the (public) scheme can decode the data instantly, so encoding provides no confidentiality
whatsoever. *Encryption* transforms data into ciphertext using an algorithm and a secret *key*; it is
reversible only by someone holding the right key, and its purpose is confidentiality. *Hashing* applies a
one-way function to produce a fixed-size digest; it is deliberately *not* reversible and uses no key (in
its plain form), and its purpose is integrity and identification, not secrecy. The table and figure below
make the contrast precise.

| Property | Encoding | Encryption | Hashing |
|---|---|---|---|
| Purpose | Usability / transport | Confidentiality | Integrity / verification |
| Uses a key? | No | Yes (secret) | No (plain) / key for MAC |
| Reversible? | Yes (public scheme) | Yes (with key) | No (one-way) |
| Output size | Varies with input | Varies with input | Fixed (e.g., 256 bits) |
| Examples | ASCII, Base64, URL-encode | AES, RSA, ChaCha20 | SHA-256, SHA-3, BLAKE2 |
| Security provided | None | Confidentiality | Integrity (not secrecy) |

The single most important takeaway is that **encoding is not encryption**: Base64-encoding a password
hides nothing, because anyone can decode it. Use encoding for compatibility, encryption for secrecy, and
hashing for integrity, and never substitute one for another.

```{image} ../../assets/figures/encoding_encryption_hashing.png
:alt: Encoding (reversible, no key), encryption (reversible with key), hashing (one-way, fixed size)
:width: 95%
:align: center
```


In [ ]:
# Chapter 2 -- Encoding vs Encryption vs Hashing (one-way vs reversible, key vs no key)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(3, 1, figsize=(9, 6.4))
def row(a, title, mid, out, back, keynote):
    def box(x,label,color,w=1.8):
        a.add_patch(FancyBboxPatch((x,0.8),w,0.8,boxstyle="round,pad=0.05",facecolor=color,edgecolor="black"))
        a.text(x+w/2,1.2,label,ha="center",va="center",fontsize=8.5)
    box(0.2,"Data","#d6eaf8"); box(3.2,mid,"#aed6f1"); box(6.4,out,"#f5b7b1")
    a.add_patch(FancyArrowPatch((2.0,1.2),(3.2,1.2),arrowstyle="->",mutation_scale=14))
    a.add_patch(FancyArrowPatch((5.0,1.2),(6.4,1.2),arrowstyle="->",mutation_scale=14))
    if back:
        a.add_patch(FancyArrowPatch((6.4,0.95),(2.0,0.95),arrowstyle="->",mutation_scale=12,color="#27ae60",
                    connectionstyle="arc3,rad=0.25"))
        a.text(4.2,0.35,"reversible "+keynote,ha="center",fontsize=8,color="#1e8449")
    else:
        a.text(8.7,1.2,"one-way\n(no inverse)",ha="center",fontsize=8,color="#c0392b")
    a.set_xlim(0,10); a.set_ylim(0,2); a.axis("off"); a.set_title(title,fontsize=10,loc="left")
row(ax[0],"Encoding (no key, reversible by anyone)","Encode\n(public scheme)","Encoded",True,"(no key)")
row(ax[1],"Encryption (secret key, reversible only with key)","Encrypt\n(+ secret key)","Ciphertext",True,"(needs key)")
row(ax[2],"Hashing (no key, NOT reversible, fixed size)","Hash fn","Digest (fixed)",False,"")
plt.tight_layout(); plt.savefig("ch02_encoding_encryption_hashing.png", dpi=110)
print("Saved ch02_encoding_encryption_hashing.png")

## 2.2 Classical Ciphers and Why They Fall

With the goals and the ground rule (Kerckhoffs's principle) in hand, the natural way to build intuition is to watch weak systems fail. The classical ciphers below show, concretely, why amateur schemes collapse, and they teach the cryptanalyst's habits we will rely on for the rest of the chapter.

The history of cryptography before the twentieth century is, in effect, a long demonstration of why
intuition is a poor guide to security. Studying classical ciphers is valuable not for their practical
use, which is nil, but because breaking them teaches the analytical habits of the cryptanalyst.

The simplest is the **Caesar cipher**, named for Julius Caesar, which shifts each letter a fixed number
of positions through the alphabet. With a shift of three, A becomes D, B becomes E, and so on. The key
is just the shift amount, of which there are only 25 useful values, so an attacker can simply try all
of them, a *brute-force* attack, and read the one result that makes sense. A cipher whose entire
keyspace can be searched in moments offers no security.

A **monoalphabetic substitution cipher** generalizes this by mapping each letter to an arbitrary other
letter, giving a keyspace of 26 factorial, about 2 to the 88th power, far too large to brute force.
Yet these ciphers fall easily to **frequency analysis**, first described by the ninth-century scholar
al-Kindi. The insight is that a substitution cipher hides *which* letter each symbol represents but
preserves *how often* each appears. Because the letter E is the most common in English, the most
frequent ciphertext symbol very likely represents E, and so on down the distribution, supplemented by
common digrams (TH, ER) and word patterns. The structure of the language leaks straight through the
encryption.

The **Vigenere cipher** attempts to defeat frequency analysis by using a repeating keyword to apply
different Caesar shifts to successive letters, flattening the frequency distribution. It resisted
analysis for centuries and was called *le chiffre indechiffrable*, the indecipherable cipher. But it
too falls: once the key length is found (by the Kasiski examination or by index-of-coincidence
methods), the ciphertext splits into separate Caesar ciphers, each broken by frequency analysis. The
deep lesson, which the one-time pad in the next section makes precise, is that a *short, repeating* key
cannot securely encrypt a *long* message, because the repetition reintroduces exploitable structure.

The code cell below provides tools to experiment with the Caesar cipher and frequency analysis. Try the
challenge ciphertexts and observe how quickly structure betrays the key.

Beyond substitution lies a second classical family, the **transposition ciphers**, which do not replace
letters but *rearrange* them according to a secret pattern, such as writing the message into a grid by
rows and reading it out by columns (the columnar transposition). Transposition preserves the letter
frequencies of the plaintext, so a frequency count alone looks like normal language; the cryptanalyst
must instead detect anagram structure and probable column orderings. Real historical systems often
combined substitution and transposition to frustrate both attacks at once. The most sophisticated
pre-computer machine, the German **Enigma** of the Second World War, used rotating wired rotors to
create a polyalphabetic substitution that changed with every keypress, yielding an astronomically large
key space. It was nonetheless broken by Allied cryptanalysts at Bletchley Park, led by Marian Rejewski's
earlier Polish work and by Alan Turing, who exploited operational mistakes, predictable message
formats, and the machine's one structural flaw (a letter never encrypted to itself) using
electromechanical "bombes." Enigma's defeat is the historical bridge to the computer age and a permanent
reminder that procedural and implementation errors, not just weak algorithms, decide real outcomes.

### Classical versus Modern Ciphers

Stepping back from the individual classical ciphers, it is worth drawing the line between the *classical*
era just surveyed and the *modern* cryptography that occupies the rest of this chapter, because the
distinction is conceptual, not merely chronological. **Classical ciphers** (Caesar, substitution,
Vigenere, transposition, and electromechanical machines such as Enigma) operate on letters and rely for
their security on the *secrecy of the method* or on modest key spaces; they fall to pencil-and-paper
techniques such as frequency analysis or, today, to a fraction of a second of computer time. **Modern
ciphers** operate on bits, assume the algorithm is fully public (Kerckhoffs's principle), and base their
security on *computational hardness*, mathematical problems believed infeasible to solve without the key,
so that breaking them is not impossible in principle but is infeasible in practice for any realistic
adversary.

| Aspect | Classical ciphers | Modern ciphers |
|---|---|---|
| Operate on | Letters / characters | Bits and bytes |
| Security basis | Secrecy of method, small key space | Public algorithm, computational hardness |
| Key sizes | Tiny (a shift, a keyword) | 128-256 bits (symmetric), 2048+ (RSA) |
| Broken by | Frequency analysis, brute force | Believed infeasible with correct use |
| Examples | Caesar, Vigenere, transposition, Enigma | AES, ChaCha20, RSA, ECC, SHA-3 |
| Performed by | Hand or simple machines | Computers (often with hardware acceleration) |

This shift, from secrecy-of-method to computational hardness over a public algorithm, is the dividing
line of the field, and it is why the rest of this chapter is organized around the mathematical problems
(factoring, discrete logarithms, and their elliptic-curve and lattice variants) that make modern
security possible, and around the computing power that threatens it, the subject revisited in Section 2.15.

### A Classification of Ciphers

It helps to see the whole landscape of ciphers in one picture. The earliest known systematic
classification of ciphers, along with the frequency-analysis technique used to break them, is credited to
the ninth-century Arab polymath **al-Kindi**, whose work makes him a founder of cryptanalysis. A modern
taxonomy extends his scheme to the computational era, organizing ciphers first by the classical-versus-
modern divide and then by mechanism.

```{mermaid}
graph TD
    C[Ciphers] --> CL[Classical]
    C --> MO[Modern]
    CL --> SUB[Substitution]
    CL --> TRANS[Transposition]
    SUB --> MONO[Monoalphabetic: Caesar, simple substitution]
    SUB --> POLY[Polyalphabetic: Vigenere, Enigma]
    TRANS --> COL[Columnar / rail-fence]
    MO --> SYM[Symmetric: one shared key]
    MO --> ASY[Asymmetric: public/private key pair]
    SYM --> BLK[Block: AES, DES/3DES]
    SYM --> STR[Stream: ChaCha20, RC4]
    ASY --> FAC[Factoring-based: RSA]
    ASY --> DLOG[Discrete-log / ECC: DH, ECDSA, EdDSA]
    MO --> PQC[Post-quantum: lattice ML-KEM/ML-DSA, hash-based]
```

Reading the tree from the top, classical ciphers split into *substitution* (replacing symbols, either
one fixed alphabet, monoalphabetic, or several, polyalphabetic) and *transposition* (rearranging symbols),
while modern ciphers split into *symmetric* (block and stream, sharing one key) and *asymmetric* (built on
factoring or discrete-logarithm hardness), with *post-quantum* families now joining the modern branch.
Every cipher named in this chapter has a place on this tree, and locating an unfamiliar algorithm on it is
the fastest way to grasp how it works and what it is good for.


In [ ]:
# Chapter 2 -- Classical ciphers and frequency analysis (self-contained)
from collections import Counter
import string

def caesar(text, shift):
    out = []
    for ch in text:
        if ch.isupper():
            out.append(chr((ord(ch) - 65 + shift) % 26 + 65))
        elif ch.islower():
            out.append(chr((ord(ch) - 97 + shift) % 26 + 97))
        else:
            out.append(ch)
    return "".join(out)

def caesar_bruteforce(cipher):
    print("All 26 Caesar shifts (look for readable English):")
    for s in range(26):
        print(f"  shift {s:2d}: {caesar(cipher, -s)[:60]}")

def letter_frequency(text):
    letters = [c.lower() for c in text if c.isalpha()]
    n = len(letters)
    freq = Counter(letters)
    print(f"Letter frequencies over {n} letters (English E~12.7%, T~9.1%, A~8.2%):")
    for ch, cnt in freq.most_common(8):
        print(f"  {ch}: {cnt:4d}  ({100*cnt/n:5.1f}%)")

# Demonstration
msg = "The quick brown fox jumps over the lazy dog near the old stone bridge."
ct = caesar(msg, 3)
print("Plaintext :", msg)
print("Caesar(+3):", ct)
print()
caesar_bruteforce("Wkh txlfn eurzq ira")
print()
# Frequency analysis on a longer substitution sample
sample = caesar(msg * 3, 7)   # stand-in for a monoalphabetic ciphertext
letter_frequency(sample)
print("\\nChallenge: cipher = 'Ro)x~)|ynwm)vx{n)xw)lxoonn)}qjw)xw)R])|nl~{r}5'  (try non-26 shifts)")

## 2.3 Perfect Secrecy and the One-Time Pad

Breaking classical ciphers raises an obvious question: is unbreakable encryption even possible? Shannon answered yes, and the answer reshapes how we think about every cipher that follows.

In 1949 Claude Shannon placed cryptography on a rigorous mathematical footing by defining **perfect
secrecy**. A cipher is perfectly secret if observing the ciphertext gives an adversary *no information
whatsoever* about the plaintext, beyond what they knew before. Formally, for every plaintext m and
every ciphertext c, the probability of m given c equals the probability of m: the ciphertext and
plaintext are statistically independent. An adversary with unlimited computing power learns nothing.

Remarkably, perfect secrecy is achievable, by the **one-time pad (OTP)**. The pad is a random key as
long as the message; encryption combines each plaintext bit with the corresponding key bit using
exclusive-or (XOR), and decryption XORs again with the same key. If the key is truly random, used only
once, kept secret, and at least as long as the message, the result is provably unbreakable. The
intuition is that for any observed ciphertext, *every* plaintext of that length is equally possible
under some key, so the ciphertext reveals nothing.

```{admonition} Going Deeper (graduate/research): why OTP is optimal and why we abandon it
:class: tip
Shannon proved a stronger negative result: perfect secrecy *requires* the key space to be at least as
large as the message space, so the key must be at least as long as the message. This is why perfect
secrecy does not scale: to send a gigabyte secretly you must first share a gigabyte of secret key,
which merely relocates the problem. Modern cryptography therefore trades Shannon's *information-theoretic*
security for *computational* security: ciphers like AES are not unbreakable in principle, but breaking
them is believed to require computational effort far beyond any feasible adversary. The formal goal
becomes *semantic security* (equivalently, indistinguishability under chosen-plaintext attack,
IND-CPA): no efficient adversary can distinguish the encryptions of two chosen messages with more than
negligible advantage. Block ciphers in a proper mode, seeded by a short key, approximate the OTP's
guarantee against bounded adversaries while keeping the key small.
```

The one-time pad also illustrates how cryptography fails in practice. The "one-time" requirement is
absolute: if the same pad encrypts two messages, XORing the two ciphertexts cancels the key and leaks
the XOR of the plaintexts, which is often enough to recover both. This exact mistake broke Soviet
traffic in the mid-twentieth century VENONA project, when pad pages were reused. The pattern, *key
reuse destroys security*, recurs throughout cryptography, from the OTP to stream ciphers to the nonce
reuse that breaks modern authenticated encryption.

```{admonition} Going Deeper (graduate/research): the semantic-security game
:class: tip
Computational security is defined through *games* between a challenger and an adversary. In the
indistinguishability under chosen-plaintext attack (IND-CPA) game, the adversary may encrypt arbitrary
messages, then submits two equal-length messages m0 and m1; the challenger secretly flips a bit b and
returns the encryption of m_b; the adversary, still able to request more encryptions, must guess b. The
scheme is IND-CPA secure if no efficient adversary guesses correctly with probability more than
negligibly above one-half. This formalism explains several earlier points at once. It is why encryption
must be *randomized* or nonce-based: a deterministic scheme loses the game instantly, because the
adversary just encrypts m0 and m1 and compares. It is why ECB fails: identical blocks are
distinguishable. And it scales up to IND-CCA, where the adversary also gets a decryption oracle, the
model that authenticated encryption is built to satisfy. Reasoning in terms of what an adversary can
*distinguish*, rather than merely what they can *read*, is the conceptual leap that separates modern
provable-security cryptography from its classical predecessors, and it is the foundation on which Boneh
and Shoup, and Katz and Lindell, build their treatments.
```


## 2.4 Randomness: True, Pseudo, and Cryptographically Secure

Perfect secrecy depended entirely on one assumption we glossed over, a truly random key. That assumption is so load-bearing, and so often violated in practice, that it deserves a section of its own before we build real ciphers on top of it.

Every cryptographic guarantee in the previous section rested on one phrase: *truly random key*.
Randomness is the silent foundation of cryptography, and weak randomness is one of the most common and
catastrophic real-world failures. We must distinguish three notions.

**True randomness** comes from physical, unpredictable processes: electronic noise, radioactive decay,
or timing jitter. Operating systems gather such *entropy* and expose it (for example through Linux
`/dev/random` and the `getrandom` system call). True entropy is the gold standard but can be slow to
collect.

**Pseudo-randomness** is produced by deterministic algorithms, *pseudo-random number generators*
(PRNGs), that stretch a small seed into a long sequence that looks random by statistical tests. The
classic example is the **linear congruential generator (LCG)**, which computes each value from the
previous one as `x = (a*x + c) mod m`. LCGs are fast and fine for simulations and games, but they are
catastrophically insecure for cryptography: given a few outputs, an attacker can solve for the internal
state and predict all past and future values. General-purpose generators such as the Mersenne Twister
(Python's default `random` module) are likewise predictable and must never be used for keys, tokens, or
nonces.

**Cryptographically secure pseudo-random number generators (CSPRNGs)** are PRNGs with an additional
guarantee: even an adversary who sees a long run of output cannot predict the next bit with advantage
better than chance, and cannot recover the internal state. In Python, `secrets` and
`random.SystemRandom` (backed by `os.urandom`) are CSPRNGs; the plain `random` module is not. The
practical rule is simple and worth memorizing: **for anything security-sensitive, use a CSPRNG**.
Predictable randomness has broken real systems repeatedly, from a 2008 Debian OpenSSL bug that
shrank the key space to a few thousand possibilities, to cryptocurrency wallets drained because their
keys were generated with weak entropy.

Where does true entropy actually come from? Operating systems harvest unpredictability from physical
events, the precise timing of interrupts, keystrokes, mouse movement, and disk activity, and many modern
CPUs include a hardware random-number generator (Intel's RDRAND, for example) seeded by thermal noise.
This entropy is collected into a pool that seeds a CSPRNG. A long-standing point of confusion on Linux is
the difference between `/dev/random` and `/dev/urandom`: historically the former could block waiting for
"fresh" entropy while the latter would not, but on modern kernels, once the pool has been initialized
once, both are cryptographically secure, and the recommended interface is the `getrandom` system call.
The real-world failures are sobering. Embedded devices and virtual machines often boot with little
entropy and have generated predictable keys at scale; a famous 2012 study found large numbers of TLS and
SSH keys sharing factors because devices generated them before gathering enough randomness. The
operational takeaways are to ensure adequate entropy at first boot, to prefer the OS CSPRNG over any
home-grown generator, and to be especially careful in virtualized and embedded environments.


In [ ]:
# Chapter 2 -- Randomness: predictable PRNG vs CSPRNG
import secrets

# A linear congruential generator (predictable!) -- glibc-style constants
class LCG:
    def __init__(self, seed): self.state = seed
    def next(self):
        self.state = (1103515245 * self.state + 12345) & 0x7fffffff
        return self.state

lcg = LCG(seed=42)
outputs = [lcg.next() for _ in range(5)]
print("LCG outputs   :", outputs)
# An attacker who knows the constants reproduces the stream exactly:
attacker = LCG(seed=42)
print("Attacker copy :", [attacker.next() for _ in range(5)])
print("=> A deterministic PRNG with a known/recovered seed is fully predictable.\\n")

# CSPRNG: unpredictable, suitable for keys and tokens
print("Secure 256-bit key   :", secrets.token_hex(32))
print("Secure URL-safe token:", secrets.token_urlsafe(24))
print("Secure dice roll 1-6 :", 1 + secrets.randbelow(6))
print("\\nRule: use `secrets` / os.urandom for keys, nonces, tokens; never `random` for security.")

## 2.5 Symmetric Encryption: Stream and Block Ciphers

Armed with secure randomness, we can finally build the practical, scalable ciphers that protect real data. We trade Shannon's perfect secrecy for computational security and gain something invaluable: short keys that encrypt long messages.

**Symmetric cryptography** uses a single shared secret key for both encryption and decryption. It is
fast and efficient, making it the workhorse for encrypting bulk data, but it raises the *key
distribution problem*: the two parties must somehow share the secret key over a secure channel before
they can communicate, a problem solved later by public-key methods. Symmetric ciphers come in two
families.

**Stream ciphers** encrypt data one bit or byte at a time by generating a pseudo-random *keystream*
from the key and XORing it with the plaintext, much like a practical approximation of the one-time pad
with a short key. They are fast and well suited to streaming data, but they are fragile: reusing a
keystream is fatal, exactly as with the OTP. Modern stream ciphers such as **ChaCha20** are widely used,
notably in TLS and in mobile devices, and the older RC4 is now deprecated because of fatal biases.

**Block ciphers** encrypt fixed-size blocks of bits at a time, typically 128 bits, transforming a
whole block under the key. The dominant block cipher is the **Advanced Encryption Standard (AES)**,
selected by the U.S. National Institute of Standards and Technology in 2001 (the Rijndael algorithm by
Daemen and Rijmen) to replace the aging Data Encryption Standard (DES). DES had a 56-bit key that
modern hardware brute-forces in hours; AES supports 128, 192, and 256-bit keys and remains secure.

Internally, AES is a *substitution-permutation network* that applies several *rounds* (10, 12, or 14
depending on key size) of four operations to a 4-by-4 byte state: SubBytes (a non-linear byte
substitution through an S-box, providing Shannon's *confusion*), ShiftRows and MixColumns (linear
mixing that spreads each byte's influence across the block, providing *diffusion*), and AddRoundKey
(XOR with a round key derived from the main key by the key schedule). Confusion obscures the
relationship between key and ciphertext; diffusion ensures that changing one plaintext bit changes
about half the ciphertext bits, the *avalanche effect*. You do not need to implement AES, modern CPUs
even have dedicated AES-NI instructions, but you must understand that a block cipher alone only encrypts
one block. To encrypt a real message of many blocks, we need a *mode of operation*, the subject of the
next section, and the choice of mode is where security is most often won or lost.

It is worth understanding the structure AES replaced, because it still appears in legacy systems and in
exam questions. The **Data Encryption Standard (DES)**, standardized in 1977, is built on a **Feistel
network**, a design that splits each block in half and, over sixteen rounds, repeatedly mixes one half
into the other using a round function and round keys. The Feistel structure has an elegant property: the
same circuitry performs encryption and decryption simply by reversing the order of round keys, which
made hardware cheap. DES's fatal weakness was not its design but its **56-bit key**, far too small;
by the late 1990s purpose-built machines and distributed efforts brute-forced DES keys in days, then
hours. The stop-gap **Triple DES (3DES)** applied DES three times with multiple keys to enlarge the
effective key length, but it is slow, has a small 64-bit block that invites birthday-bound attacks on
large data volumes (the Sweet32 attack), and is now deprecated. AES, a substitution-permutation network
rather than a Feistel cipher, with 128-bit blocks and 128- to 256-bit keys, resolved all of these
issues and is the symmetric standard today. The lesson for selecting cryptography is concrete: key
length and block size are not academic parameters but the difference between secure and broken.

Before detailing block ciphers, it is worth fixing the broader trade-off that explains why symmetric
cryptography remains indispensable despite the elegance of public-key methods covered later. Symmetric
ciphers are typically several orders of magnitude faster than asymmetric ones and produce no ciphertext
expansion to speak of, which is why essentially all bulk data, disk volumes, network streams, database
fields, is protected symmetrically. The price is the key-distribution problem, and the standard
architecture resolves it through *hybrid encryption*: use a slow public-key operation once, only to
establish or transport a fresh symmetric session key, then encrypt all the actual data with that fast
symmetric key. TLS, encrypted email, and messaging apps all follow this pattern, and recognizing it now
will make the later sections on RSA, Diffie-Hellman, and TLS fall into place as solutions to the single
problem this section sets up.

```{image} ../../assets/figures/symmetric.png
:alt: Symmetric encryption: plaintext is encrypted and decrypted with the same shared secret key
:width: 95%
:align: center
```

In symmetric encryption, the *same secret key* is used for both encryption and decryption, as the diagram
shows. Alice and Bob must both hold the identical key, which is why secure key distribution is the central
challenge.

```{mermaid}
flowchart LR
    P1[Plaintext] -->|Encrypt| C[Ciphertext]
    C -->|Decrypt| P2[Plaintext]
    K([Shared secret key]) -.-> |used to encrypt| C
    K -.-> |same key used to decrypt| P2
```


In [ ]:
# Chapter 2 -- Illustration: symmetric encryption (one shared key)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(figsize=(9, 3.2))
def box(x, label, color):
    ax.add_patch(FancyBboxPatch((x, 0.9), 1.6, 0.8, boxstyle="round,pad=0.05",
                facecolor=color, edgecolor="black"))
    ax.text(x+0.8, 1.3, label, ha="center", va="center", fontsize=9)
box(0.2, "Plaintext", "#d6eaf8"); box(3.0, "Encrypt", "#aed6f1")
box(5.8, "Ciphertext", "#f5b7b1"); box(8.6, "Decrypt", "#aed6f1")
box(11.4, "Plaintext", "#d6eaf8")
for x0 in (1.8, 4.6, 7.4, 10.2):
    ax.add_patch(FancyArrowPatch((x0, 1.3), (x0+1.2, 1.3), arrowstyle="->", mutation_scale=16))
# same key feeding both encrypt and decrypt
ax.text(3.8, 2.6, "SHARED SECRET KEY", ha="center", fontsize=9, fontweight="bold", color="#b9770e")
ax.add_patch(FancyArrowPatch((3.8, 2.5), (3.8, 1.75), arrowstyle="->", mutation_scale=14, color="#b9770e"))
ax.add_patch(FancyArrowPatch((3.8, 2.5), (9.4, 1.75), arrowstyle="->", mutation_scale=14, color="#b9770e"))
ax.text(6.5, 0.4, "Same key encrypts and decrypts (e.g., AES-GCM, ChaCha20). Fast; key distribution is the challenge.",
        ha="center", fontsize=8, style="italic")
ax.set_xlim(0, 13.2); ax.set_ylim(0, 3); ax.axis("off")
ax.set_title("Symmetric Encryption")
plt.tight_layout(); plt.savefig("ch02_symmetric.png", dpi=110)
print("Saved ch02_symmetric.png")

## 2.6 Block Cipher Modes of Operation

A block cipher by itself only transforms a single block, which is almost never what we need. How we stitch many blocks together, the mode of operation, turns out to matter as much as the cipher, and is where security is most often quietly lost.

A block cipher encrypts exactly one block. A *mode of operation* specifies how to use the cipher
repeatedly to encrypt a message longer than one block, and this seemingly mechanical choice has
enormous security consequences.

The naive approach is **Electronic Codebook (ECB)** mode: split the plaintext into blocks and encrypt
each independently with the same key. ECB is simple and parallelizable, and it is also broken, because
identical plaintext blocks produce identical ciphertext blocks. The result leaks the structure of the
data. The famous demonstration encrypts a bitmap image of the Linux penguin in ECB mode: the outline
of the penguin remains plainly visible in the ciphertext, because regions of identical color encrypt to
identical patterns. **Never use ECB for data with any structure**, which is essentially all real data.

Secure modes break this pattern by chaining or randomizing each block. **Cipher Block Chaining (CBC)**
XORs each plaintext block with the previous ciphertext block before encrypting, and uses a random
*initialization vector (IV)* for the first block, so identical plaintexts encrypt differently each
time. CBC requires padding and is sequential for encryption. **Cipher Feedback (CFB)** and **Output
Feedback (OFB)** turn a block cipher into a stream cipher. **Counter (CTR)** mode encrypts an
incrementing counter to produce a keystream that is XORed with the plaintext; it is parallelizable,
needs no padding, and is widely preferred, but it is fatally insecure if a counter/nonce value is ever
reused with the same key.

Crucially, all of the modes above provide *confidentiality only*. They do not detect tampering: an
attacker can flip bits in the ciphertext and cause predictable changes in the decrypted plaintext. This
gap is closed by **Galois/Counter Mode (GCM)**, an *authenticated encryption* mode that combines CTR-mode
confidentiality with a built-in authentication tag, so any modification is detected on decryption.
AES-GCM and the stream-cipher construction ChaCha20-Poly1305 are the modern defaults; they belong to
the AEAD family discussed in Section 2.8. The code cell below makes ECB's failure visible by encrypting
a simple patterned image in ECB and CBC modes.

A practical note ties these modes to engineering reality. CTR and GCM are *parallelizable* and require no
padding, which suits high-throughput systems and is one reason AES-GCM dominates TLS; CBC is sequential
and needs padding, which historically exposed it to padding-oracle attacks. All modes that take an IV or
nonce demand care: the IV for CBC must be unpredictable, while the nonce for CTR and GCM must be unique
but need not be secret, and confusing these requirements has broken real systems. The single most
important operational rule across every mode is the one stated for the one-time pad and repeated for GCM:
the combination of key and nonce must never repeat. When this discipline is hard to guarantee, for
example across distributed servers, prefer a nonce-misuse-resistant mode such as AES-GCM-SIV, which
degrades gracefully if a nonce is accidentally reused.


In [ ]:
# Chapter 2 -- ECB vs CBC: why ECB leaks structure (self-contained)
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

# Build a simple structured image (large blocks of constant color)
H = W = 128
img = np.zeros((H, W), dtype=np.uint8)
img[20:108, 20:60] = 200          # a bright rectangle
img[40:90, 70:108] = 120          # a mid-gray rectangle
raw = img.tobytes()

key, iv = os.urandom(16), os.urandom(16)

def aes_encrypt(data, mode):
    padder = padding.PKCS7(128).padder()
    padded = padder.update(data) + padder.finalize()
    enc = Cipher(algorithms.AES(key), mode).encryptor()
    return enc.update(padded) + enc.finalize()

ecb = aes_encrypt(raw, modes.ECB())[:H*W]
cbc = aes_encrypt(raw, modes.CBC(iv))[:H*W]
ecb_img = np.frombuffer(ecb, dtype=np.uint8).reshape(H, W)
cbc_img = np.frombuffer(cbc, dtype=np.uint8).reshape(H, W)

fig, ax = plt.subplots(1, 3, figsize=(11, 4))
for a, im, t in zip(ax, [img, ecb_img, cbc_img],
                    ["Original", "AES-ECB (structure leaks!)", "AES-CBC (looks random)"]):
    a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.savefig("ch02_ecb_vs_cbc.png", dpi=110)
print("Saved ch02_ecb_vs_cbc.png")
print("Note how the ECB ciphertext preserves the rectangles, while CBC does not.")

```{image} ../../assets/figures/ch02_ecb_vs_cbc.png
:alt: ECB leaks image structure while CBC produces random-looking ciphertext
:width: 90%
:align: center
```


## 2.7 Cryptographic Hash Functions

So far we have pursued confidentiality. But security also demands knowing that data has not changed, and that requires a different primitive entirely. Hash functions give us a compact fingerprint of data, the foundation for integrity, signatures, and much more.

A **cryptographic hash function** takes an input of any size and produces a fixed-size output called a
*digest* or *hash*. Examples include SHA-256 (a 256-bit digest from the SHA-2 family) and SHA-3.
Hashes are everywhere: they verify file integrity, index data, store passwords (with the extra steps in
Section 2.9), underpin digital signatures, and form the backbone of blockchains. A function suitable for
cryptography must satisfy three security properties.

**Preimage resistance** (one-wayness): given a digest h, it is computationally infeasible to find any
input m such that hash(m) = h. You cannot run the function backward. **Second-preimage resistance**:
given a specific input m1, it is infeasible to find a different input m2 with the same digest.
**Collision resistance**: it is infeasible to find *any* two distinct inputs with the same digest.
Collisions must exist mathematically, because infinitely many inputs map to finitely many digests, but
a good hash makes finding them infeasible. A related requirement is the *avalanche effect*: changing a
single input bit should change roughly half the output bits, so digests of similar inputs look totally
unrelated.

The security of collision resistance is bounded by the **birthday paradox**: because of the
probabilistic ease of finding *some* matching pair, an attacker needs only about 2 to the n/2 work to
find a collision in an n-bit hash, not 2 to the n. This is why a 256-bit hash offers only about 128
bits of collision security, and why digest sizes are chosen generously. History shows the stakes: MD5
and SHA-1 were once standard but are now broken for collision resistance (practical MD5 collisions were demonstrated by Xiaoyun Wang and Hongbo Yu in 2004-2005, and researchers produced a full SHA-1 collision in 2017), so neither may be used where collision resistance matters, such as
certificates or signatures. Use SHA-256 or stronger today.

Most classical hashes (MD5, SHA-1, SHA-2) are built with the **Merkle-Damgard construction**, which
processes the message in fixed-size blocks through a compression function. This design has a quirk, the
*length-extension attack*, in which an attacker who knows hash(m) can compute hash(m || extra) without
knowing m, which is why naive `hash(secret || message)` must never be used as a MAC; HMAC (Section 2.8)
fixes this. SHA-3 uses a different *sponge* construction that is immune to length extension. A related
structure, the **Merkle tree**, hashes data in a tree so that any single item can be verified against a
small root hash, a technique central to Git, certificate transparency, and blockchains. The code cell
demonstrates the avalanche effect and a simple integrity check.

Because they are fast, deterministic, and collision-resistant, hash functions appear far beyond
integrity checking, and recognizing these uses helps connect later chapters. They enable **deduplication
and content addressing** (Git names every object by its SHA hash, so identical content is stored once and
any tampering is detectable). They provide **commitment**: publishing the hash of a value lets you reveal
the value later and prove you had not changed it, the basis of many protocols. They drive **proof of
work** in blockchains, where miners search for inputs whose hash meets a difficulty target. They support
**password-adjacent** uses such as deriving lookup keys and HMAC-based one-time passwords. And the
**Merkle tree** structure, by letting any single leaf be verified against a tiny root hash with a short
"proof path," underpins Certificate Transparency, peer-to-peer file distribution, and the tamper-evident
ledgers of distributed systems. When you meet these mechanisms in later chapters, remember that a single
primitive, the cryptographic hash, is doing the heavy lifting.


In [ ]:
# Chapter 2 -- Hash functions: avalanche effect and integrity check
import hashlib

def sha256(s): return hashlib.sha256(s.encode()).hexdigest()

a = sha256("The quick brown fox")
b = sha256("The quick brown fox.")   # one extra character
print("SHA-256('...fox') :", a)
print("SHA-256('...fox.'):", b)
# Hamming distance between the two digests (in bits)
xa, xb = int(a,16), int(b,16)
print("Bits changed by a 1-char edit:", bin(xa ^ xb).count("1"), "of 256  (avalanche)\\n")

# Integrity verification: detect tampering
message = "Transfer $100 to Alice"
digest  = sha256(message)
received = "Transfer $900 to Alice"   # attacker altered the amount
print("Stored digest   :", digest)
print("Recomputed match:", sha256(received) == digest, "=> tampering detected")

# Compare digest sizes / algorithms
for algo in ["md5", "sha1", "sha256", "sha3_256"]:
    h = hashlib.new(algo); h.update(b"hello")
    print(f"  {algo:9s} ({h.digest_size*8:3d} bits): {h.hexdigest()[:32]}...")

## 2.8 Message Authentication Codes and Authenticated Encryption

A hash alone proves a message is unchanged only if the digest itself cannot be forged, which fails the moment an attacker can recompute it. Adding a secret key closes that gap and lets us combine integrity with confidentiality.

Encryption hides data but, as we saw, most modes do not detect tampering. **Integrity** and
**authentication** are provided by a **message authentication code (MAC)**: a short tag computed from
the message and a shared secret key. The sender transmits the message and tag; the receiver recomputes
the tag with the same key and accepts only if it matches. Because the attacker lacks the key, they
cannot forge a valid tag for a modified message. A MAC thus proves both that the message is unchanged
(integrity) and that it came from someone holding the key (authentication), though, unlike a signature,
it does not provide non-repudiation, since either party could have produced the tag.

The standard MAC is **HMAC** (hash-based MAC), which wraps a hash function in a specific keyed
construction, `HMAC(K, m) = H((K xor opad) || H((K xor ipad) || m))`. This nested design makes HMAC
secure even when built on a Merkle-Damgard hash, immunizing it against the length-extension attack that
dooms naive `H(K || m)`. HMAC-SHA256 is ubiquitous in APIs, tokens, and protocols.

Combining encryption and a MAC correctly is subtle, and doing it by hand invites mistakes (the order
matters; *encrypt-then-MAC* is the safe composition). The modern solution is **authenticated encryption
with associated data (AEAD)**, which provides confidentiality, integrity, and authentication in a single
primitive, and also authenticates *associated data* (such as packet headers) that must be visible but
unmodified. The two dominant AEAD constructions are **AES-GCM** and **ChaCha20-Poly1305**; both are used
in TLS 1.3. The non-negotiable rule for AEAD is *never reuse a nonce with the same key*: nonce reuse in
GCM not only leaks plaintext relationships but can expose the authentication key itself, enabling
forgeries. When in doubt, prefer a misuse-resistant AEAD or a library that manages nonces for you.

```{admonition} Going Deeper (graduate/research): security definitions
:class: tip
The relevant security notion for authenticated encryption is *indistinguishability under chosen-
ciphertext attack* (IND-CCA) combined with *ciphertext integrity* (INT-CTXT); together these imply
authenticated-encryption security (AE). A scheme that is IND-CPA only (like raw CBC) succumbs to
chosen-ciphertext attacks such as the *padding oracle* attack, where the error behavior of a decryptor
leaks one plaintext byte at a time. The padding-oracle attack (Vaudenay, 2002) broke many real CBC
deployments and is the practical reason the field moved decisively to AEAD. The lesson is definitional:
choosing the right *security goal* (AE/IND-CCA, not merely confidentiality) is as important as choosing
a strong cipher.
```

A subtle but important question is *composition order* when building authenticated encryption from
separate encryption and MAC primitives, a topic that has caused real vulnerabilities. There are three
options: Encrypt-and-MAC (MAC the plaintext, used by SSH), MAC-then-Encrypt (MAC the plaintext, then
encrypt both, used by older TLS), and Encrypt-then-MAC (encrypt, then MAC the ciphertext, used by IPsec).
Cryptographers showed that only **Encrypt-then-MAC** is generically secure, because it lets the receiver
verify integrity *before* doing any decryption, so malformed ciphertexts are rejected without ever
exposing decryption behavior to the attacker, which is exactly what defeats padding-oracle attacks. This
is one more reason to prefer a vetted AEAD construction that gets the composition right internally rather
than assembling encryption and MAC by hand. It also illustrates a meta-lesson of the chapter: the
*details of how primitives are combined* are themselves security-critical, not mere engineering trivia.

### Hash, MAC, and Digital Signature Compared

A frequent and consequential misconception deserves to be confronted directly: **a hash by itself does not
provide integrity against a deliberate attacker.** If a message travels with a plain hash of itself, an
active attacker who can modify the message in transit can simply recompute the hash of the altered message
and attach it, and the recipient, recomputing the hash, finds a perfect match. The plain hash detects
*accidental* corruption (a flipped bit from a noisy link) but not *malicious* tampering, because nothing
about the hash is secret. Integrity against an adversary requires binding the data to a *secret* (a key)
or to an *identity* (a key pair), which is exactly what a MAC and a digital signature add.

The three primitives form a ladder of guarantees. A **hash** maps data to a fixed digest using no key; it
gives integrity only against accidental change. A **MAC** computes a tag using a *shared secret key*, so an
attacker without the key cannot forge a valid tag; it provides integrity and authentication, but because
both parties share the key, it cannot prove which party produced the tag, so it offers no non-repudiation.
A **digital signature** uses the signer's *private key*, verifiable by anyone with the corresponding public
key; it provides integrity, authentication, and non-repudiation, since only the private-key holder could
have produced the signature. The table and diagram make the distinctions precise.

| Property | Hash | MAC | Digital Signature |
|---|---|---|---|
| Key used | none | shared secret key | private key (verify with public key) |
| Integrity (accidental) | yes | yes | yes |
| Integrity (malicious) | **no** (attacker recomputes) | yes | yes |
| Authentication (origin) | no | yes (holder of shared key) | yes (specific signer) |
| Non-repudiation | no | no (either party could forge) | **yes** |
| Typical algorithms | SHA-256, SHA-3, BLAKE2 | HMAC, CMAC, Poly1305 | RSA-PSS, ECDSA, EdDSA |
| Example use | file checksum, dedup | API request authentication | code signing, certificates, contracts |

```{mermaid}
graph TD
    subgraph Hash [Hash: no key]
    H1[message] --> H2[hash fn] --> H3[digest]
    end
    subgraph MAC [MAC: shared secret key]
    M1[message + shared key] --> M2[HMAC] --> M3[tag: integrity + auth]
    end
    subgraph DS [Digital Signature: key pair]
    D1[message + signer PRIVATE key] --> D2[sign] --> D3[signature]
    D3 --> D4[verify with signer PUBLIC key: integrity + auth + non-repudiation]
    end
```

The practical rule follows directly: use a plain hash only for non-adversarial integrity (detecting
corruption or deduplicating); use a MAC when two parties share a key and need to verify that messages
between them are authentic and unmodified; and use a digital signature when verification must be public or
when non-repudiation is required, as in software updates, TLS certificates, and signed documents. This is
also why the authenticated encryption of the previous pages pairs encryption with a MAC or signature
rather than a bare hash.


## 2.9 Key Derivation and Password Storage

Keys and the passwords behind them are only as strong as how we derive and store them. Having built encryption and authentication, we now confront the very human problem of turning weak passwords into strong keys without handing attackers an easy target.

Two recurring problems require turning weak or raw key material into strong cryptographic keys: deriving
keys from human passwords, and storing passwords for authentication. Both are solved by **key derivation
functions (KDFs)**, but with an important twist.

Passwords are low-entropy: people choose predictable strings, and the space of likely passwords is
small enough to enumerate. If a site stored passwords in plaintext, a database breach would expose them
all; if it stored a fast hash like SHA-256, an attacker with the stolen hashes could try billions of
guesses per second on a GPU and crack most of them, especially with precomputed *rainbow tables*. The
defenses are *salting* and *key stretching*. A **salt** is a unique random value stored alongside each
hash; it ensures identical passwords produce different stored values and defeats precomputed tables.
**Key stretching** deliberately makes the function slow and resource-intensive, so each guess costs the
attacker dearly while remaining tolerable for a single legitimate login.

Password-hashing KDFs implement these ideas. **PBKDF2** applies a hash thousands or millions of times;
it is widely supported but only CPU-hard, so GPUs and custom hardware accelerate attacks. **bcrypt** is
deliberately slow and somewhat memory-using. **scrypt** and, the current best practice, **Argon2** (the
2015 Password Hashing Competition winner) are *memory-hard*: they require large amounts of memory,
which neutralizes the parallelism advantage of GPUs and ASICs. For deriving an encryption key from a
shared secret that is already high-entropy (for instance the output of a key exchange), a fast KDF such
as **HKDF** is appropriate; HKDF is not for passwords. The practical guidance: never store passwords
reversibly or with a plain fast hash; use Argon2id (or scrypt/bcrypt) with a unique salt and tuned cost
parameters. The code cell contrasts an insecure approach with a correct one.

Two refinements deserve mention because they appear in practice and on examinations. A **pepper** is a
secret value added to the password before hashing, like a salt but kept *separate* from the database
(for example in application configuration or an HSM), so that a database breach alone does not give the
attacker everything needed to start cracking. Unlike salts, a pepper is shared across users and must
remain secret. The second is breach response: when a password database is exposed, the correct reaction
depends on how it was stored. If passwords were stored with a strong salted, memory-hard KDF, attackers
still face enormous cost per guess, buying defenders time to force resets; if they were stored with a
fast hash or, worse, in plaintext, the credentials must be considered immediately compromised, including
everywhere users reused them, which is why *credential stuffing* (replaying leaked passwords against
other sites) is among the most common real-world attacks. These practices link cryptographic storage to
the incident-response and identity-management chapters, where the organizational response to credential
exposure is developed in full.


In [ ]:
# Chapter 2 -- Password storage: the wrong way and a better way
import hashlib, os, hmac

password = "correct horse battery staple"

# WRONG: a single fast hash, no salt -> vulnerable to rainbow tables and fast GPU cracking
bad = hashlib.sha256(password.encode()).hexdigest()
print("INSECURE sha256(pw):", bad)

# BETTER: unique salt + key stretching (PBKDF2 shown for portability; prefer Argon2id in production)
salt = os.urandom(16)
iterations = 200_000
derived = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations)
print("Salt (hex)         :", salt.hex())
print("PBKDF2 derived hash:", derived.hex())

# Verification at login: recompute with the stored salt and compare in constant time
def verify(candidate, salt, iterations, stored):
    test = hashlib.pbkdf2_hmac("sha256", candidate.encode(), salt, iterations)
    return hmac.compare_digest(test, stored)   # constant-time compare avoids timing leaks

print("Login correct pw   :", verify(password, salt, iterations, derived))
print("Login wrong   pw   :", verify("guess", salt, iterations, derived))
print("\\nProduction note: prefer argon2-cffi (Argon2id), which is memory-hard.")

## 2.10 Public-Key Cryptography and RSA

Everything so far assumed the two parties already shared a secret key, which simply relocates the problem. Public-key cryptography dissolves it, and we start with the system that made the idea concrete.

We begin where all of security ultimately bottoms out. Before any firewall or password matters, we need the mathematics that makes secrecy and trust possible at all, so this section defines what cryptography promises and, just as importantly, what it does not.

Symmetric cryptography has an inescapable bootstrapping problem: the parties must already share a secret
key. In 1976 Whitfield Diffie and Martin Hellman published the idea of **public-key (asymmetric)
cryptography**, which dissolves this problem. Each party holds a mathematically linked *key pair*: a
**public key** that may be freely published, and a **private key** kept secret. Anyone can encrypt a
message with the recipient's public key, but only the holder of the matching private key can decrypt it.
Conversely, the private key can *sign* data in a way that anyone can verify with the public key, giving
authentication and non-repudiation. Public-key cryptography rests on *trapdoor* mathematical problems:
operations easy to perform but infeasible to reverse without secret information.

The first and still most recognizable public-key system is **RSA** (Rivest, Shamir, Adleman, 1977). Its
security rests on the difficulty of *factoring* the product of two large primes. Key generation chooses
two large secret primes p and q, computes the modulus n = p*q and Euler's totient phi = (p-1)(q-1),
selects a public exponent e coprime to phi (commonly 65537), and computes the private exponent d as the
modular inverse of e modulo phi. The public key is (n, e); the private key is d. Encryption of a message
m is c = m^e mod n, and decryption is m = c^d mod n. The correctness follows from Euler's theorem; the
security follows from the belief that recovering d from the public (n, e) requires factoring n, which is
infeasible for sufficiently large n (today at least 2048 bits, with 3072 or more recommended for
long-term use). In practice RSA is never used on raw messages; secure padding (OAEP for encryption, PSS
for signatures) is essential, and "textbook RSA" without padding is insecure. The code cell illustrates
RSA on tiny numbers for understanding, then with a real library.

Two cautions make RSA safe in practice, and both are frequent exam and audit topics. First, **textbook
RSA is insecure** and must never be used directly. Because plain RSA is deterministic, identical messages
encrypt identically (leaking equality), small messages with a small public exponent can be recovered by
simply taking an integer root (m^e may be less than n, so no modular reduction occurs), and the
multiplicative homomorphism enables chosen-ciphertext manipulation. Secure padding fixes these: **OAEP**
randomizes and structures the plaintext before encryption, and **PSS** does so for signatures, which is
why every real library applies them. Second, RSA has been undone by *parameter* mistakes rather than by
factoring: reusing a modulus across users, choosing primes that are too close together, using a tiny
private exponent, or, as Section 2.4 noted, generating primes with poor randomness so that two keys share
a common factor that a simple GCD computation reveals. These lessons reinforce the chapter's refrain:
even a sound algorithm fails when its preconditions, randomness, padding, and parameter hygiene, are
violated.

```{image} ../../assets/figures/asymmetric.png
:alt: Asymmetric encryption: encrypt with the recipient public key, decrypt with the recipient private key
:width: 95%
:align: center
```

In asymmetric (public-key) encryption, a message is encrypted with the recipient's *public* key and can
be decrypted only with their matching *private* key, so anyone can send a confidential message to the
key-pair's owner without sharing a secret in advance.

```{mermaid}
flowchart LR
    P1[Plaintext] -->|Encrypt| C[Ciphertext]
    C -->|Decrypt| P2[Plaintext]
    PUB([Recipient PUBLIC key]) -.-> |encrypt| C
    PRIV([Recipient PRIVATE key]) -.-> |decrypt| P2
```

For digital signatures the key roles are reversed: the signer uses their *private* key to sign, and
anyone can verify with the signer's *public* key, providing integrity, authenticity, and non-repudiation.

```{image} ../../assets/figures/digital_signature.png
:alt: Digital signature: sign with the signer private key, verify with the signer public key
:width: 95%
:align: center
```


In [ ]:
# Chapter 2 -- Illustration: asymmetric encryption and signing (key pair)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(2, 1, figsize=(9, 5.2))
def draw(a, title, k1, k2, note):
    def box(x, label, color):
        a.add_patch(FancyBboxPatch((x, 0.9), 1.6, 0.8, boxstyle="round,pad=0.05",
                    facecolor=color, edgecolor="black"))
        a.text(x+0.8, 1.3, label, ha="center", va="center", fontsize=9)
    box(0.2,"Plaintext","#d6eaf8"); box(3.0,"Encrypt","#aed6f1")
    box(5.8,"Ciphertext","#f5b7b1"); box(8.6,"Decrypt","#aed6f1"); box(11.4,"Plaintext","#d6eaf8")
    for x0 in (1.8,4.6,7.4,10.2):
        a.add_patch(FancyArrowPatch((x0,1.3),(x0+1.2,1.3),arrowstyle="->",mutation_scale=16))
    a.text(3.8,2.6,k1,ha="center",fontsize=9,fontweight="bold",color="#1f6f8b")
    a.add_patch(FancyArrowPatch((3.8,2.5),(3.8,1.75),arrowstyle="->",mutation_scale=14,color="#1f6f8b"))
    a.text(9.4,2.6,k2,ha="center",fontsize=9,fontweight="bold",color="#7d3c98")
    a.add_patch(FancyArrowPatch((9.4,2.5),(9.4,1.75),arrowstyle="->",mutation_scale=14,color="#7d3c98"))
    a.text(6.5,0.35,note,ha="center",fontsize=8,style="italic")
    a.set_xlim(0,13.2); a.set_ylim(0,3); a.axis("off"); a.set_title(title)
draw(ax[0],"Asymmetric Encryption (confidentiality)","RECIPIENT PUBLIC KEY","RECIPIENT PRIVATE KEY",
     "Anyone encrypts with the public key; only the private-key holder can decrypt.")
draw(ax[1],"Digital Signature (authenticity, integrity, non-repudiation)","SIGNER PRIVATE KEY","SIGNER PUBLIC KEY",
     "Signer signs with the private key; anyone verifies with the public key. (Roles reversed vs. encryption.)")
plt.tight_layout(); plt.savefig("ch02_asymmetric.png", dpi=110)
print("Saved ch02_asymmetric.png")

In [ ]:
# Chapter 2 -- RSA from first principles (tiny, for learning) and with a real library
# --- Educational toy RSA (DO NOT use small numbers in practice) ---
def egcd(a, b):
    if b == 0: return (a, 1, 0)
    g, x, y = egcd(b, a % b)
    return (g, y, x - (a // b) * y)
def modinv(a, m):
    g, x, _ = egcd(a, m)
    return x % m

p, q = 61, 53
n = p * q
phi = (p - 1) * (q - 1)
e = 17
d = modinv(e, phi)
print(f"p={p} q={q}  n={n}  phi={phi}  e={e}  d={d}")
m = 42
c = pow(m, e, n)            # encryption: c = m^e mod n
m2 = pow(c, d, n)           # decryption: m = c^d mod n
print(f"message={m}  cipher={c}  decrypted={m2}  ok={m==m2}\\n")

# --- Real RSA with OAEP padding and PSS signatures ---
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
priv = rsa.generate_private_key(public_exponent=65537, key_size=2048)
pub = priv.public_key()
msg = b"Public-key cryptography solves key distribution."
ct = pub.encrypt(msg, padding.OAEP(mgf=padding.MGF1(hashes.SHA256()),
                                   algorithm=hashes.SHA256(), label=None))
pt = priv.decrypt(ct, padding.OAEP(mgf=padding.MGF1(hashes.SHA256()),
                                   algorithm=hashes.SHA256(), label=None))
print("Decrypted matches original:", pt == msg)
sig = priv.sign(msg, padding.PSS(mgf=padding.MGF1(hashes.SHA256()),
                                 salt_length=padding.PSS.MAX_LENGTH), hashes.SHA256())
print("Signature length (bytes):", len(sig), "(verifiable with the public key)")

## 2.11 Diffie-Hellman Key Exchange

RSA lets us encrypt to a public key, but asymmetric operations are slow, so in practice we use them mainly to agree on a fast symmetric key. The most elegant way to do that, over a fully public channel, is the next topic.

Public-key encryption lets two strangers communicate secretly, but asymmetric operations are slow, so in
practice we use them only to establish a shared symmetric key and then switch to fast symmetric
encryption. The original and most elegant mechanism for agreeing on a shared secret over a public channel
is the **Diffie-Hellman key exchange (DHKE)**. Its near-magical property is that two parties can derive a
common secret while an eavesdropper who sees every message exchanged still cannot compute it.

The classic construction works in modular arithmetic. The parties publicly agree on a large prime p and
a generator g. Alice picks a secret a and sends A = g^a mod p; Bob picks a secret b and sends B = g^b mod
p. Alice computes B^a mod p and Bob computes A^b mod p; both equal g^(ab) mod p, the shared secret. The
eavesdropper sees g, p, A, and B but cannot recover the secret without solving the **discrete logarithm
problem**, finding a from g^a mod p, which is infeasible for large p. The shared value is then fed through
a KDF to produce symmetric keys.

A vital caveat: plain Diffie-Hellman provides no authentication, so it is vulnerable to a
*man-in-the-middle* attack in which an active attacker performs separate exchanges with each party and
relays traffic. DH must therefore be combined with authentication (signatures or certificates), as TLS
does. A further refinement is *ephemeral* Diffie-Hellman (DHE, or ECDHE on elliptic curves), which uses
fresh random secrets for every session. This provides **forward secrecy**: even if a server's long-term
private key is later stolen, past recorded sessions remain secret, because the per-session DH secrets
were never stored. Forward secrecy is now a standard requirement and is mandatory in TLS 1.3.

To see why authentication is non-negotiable, trace the man-in-the-middle attack on unauthenticated
Diffie-Hellman concretely. Alice sends A = g^a to Bob, but the attacker Mallory intercepts it and sends
her own M = g^m to Bob; Bob replies with B = g^b, which Mallory again intercepts, sending her own value
back to Alice. Now Alice has unknowingly established a shared secret g^(am) with Mallory, and Bob has
established g^(bm) with Mallory, while each believes they are talking to the other. Mallory sits in the
middle, decrypting, reading, and re-encrypting every message, completely transparent to both parties.
Nothing in the mathematics of Diffie-Hellman detects this, because the protocol guarantees only that two
parties share a secret, not *which* two parties. The fix, used by TLS, SSH, and Signal alike, is to
authenticate the exchange: each side signs its key share, or the keys are bound to certificates or to
pre-verified identities, so Mallory cannot substitute her own values without detection. This is the
single most important practical lesson about key exchange and a frequent source of real vulnerabilities
in custom protocols that "use Diffie-Hellman" but forget to authenticate it.

### ElGamal Encryption

Diffie-Hellman lets two parties agree on a secret, but it is not by itself an encryption scheme. The
**ElGamal cryptosystem** (Taher ElGamal, 1985) turns the same discrete-logarithm hardness into full
public-key encryption, and it is both historically important and still used today (notably as the basis
of encryption in the OpenPGP standard and, in elliptic-curve form, in many protocols). Understanding it
also illuminates why public-key encryption is usually *randomized*.

The setup reuses Diffie-Hellman's parameters: a large prime p and a generator g. A user picks a private
key x and publishes the public key h = g^x mod p (along with p and g). To encrypt a message m (an integer
modulo p), the sender chooses a fresh random *ephemeral* value k and computes the ciphertext pair
(c1, c2) = (g^k mod p, m * h^k mod p). To decrypt, the holder of x computes the shared value s = c1^x mod
p (which equals g^(kx), exactly the Diffie-Hellman secret) and recovers m = c2 * s^(-1) mod p. Two
properties stand out. First, ElGamal is *probabilistic*: because a fresh random k is used every time, the
same plaintext encrypts to different ciphertexts, which is necessary for semantic security. Second, and
mirroring the one-time-pad and nonce lessons of this chapter, **k must be random, secret, and never
reused**: reusing k across two messages lets an attacker recover the ratio of the plaintexts, and a
predictable k leaks the message entirely. ElGamal is also *multiplicatively homomorphic*, a property
exploited in some privacy-preserving protocols. The worked example below implements textbook ElGamal on
small numbers, and a production-grade reference using the Crypto++ library (2048-bit keys) accompanies
this chapter's materials.


In [ ]:
# Chapter 2 -- Textbook ElGamal encryption (small numbers for learning)
import secrets

def egcd(a, b):
    if b == 0: return (a, 1, 0)
    g, x, y = egcd(b, a % b); return (g, y, x - (a // b) * y)
def modinv(a, m):
    return egcd(a, m)[1] % m

# Public parameters (tiny prime for illustration; real systems use >=2048-bit p)
p, g = 2357, 2
# Key generation
x = secrets.randbelow(p - 3) + 2      # private key
h = pow(g, x, p)                       # public key component
print(f"Public key : (p={p}, g={g}, h={h})    Private key: x={x}")

# Encryption of message m
m = 2035
k = secrets.randbelow(p - 3) + 2      # fresh ephemeral key -- NEVER reuse
c1 = pow(g, k, p)
c2 = (m * pow(h, k, p)) % p
print(f"Plaintext m = {m}  ->  ciphertext (c1={c1}, c2={c2})")

# Decryption
s = pow(c1, x, p)                      # shared secret g^(kx)
m_rec = (c2 * modinv(s, p)) % p
print(f"Decrypted  = {m_rec}   correct = {m_rec == m}")

# Demonstrate probabilistic encryption: same m, new k -> different ciphertext
k2 = secrets.randbelow(p - 3) + 2
print("Same m, different k gives:", (pow(g,k2,p), (m*pow(h,k2,p))%p))
print("=> ElGamal is randomized; reusing k would break it (analogous to nonce reuse).")

In [ ]:
# Chapter 2 -- Diffie-Hellman shared-secret derivation (small numbers for clarity)
p = 0xFFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD1  # a (small, demo) prime
g = 2
import secrets
a = secrets.randbelow(p - 2) + 2     # Alice's private secret
b = secrets.randbelow(p - 2) + 2     # Bob's private secret
A = pow(g, a, p)                     # Alice -> Bob (public)
B = pow(g, b, p)                     # Bob -> Alice (public)
alice_shared = pow(B, a, p)          # Alice computes B^a
bob_shared   = pow(A, b, p)          # Bob computes A^b
print("Alice and Bob agree:", alice_shared == bob_shared)
print("Shared secret (hex):", hex(alice_shared)[:50], "...")
print("An eavesdropper sees g, p, A, B but must solve discrete log to get the secret.")

## 2.12 Elliptic-Curve Cryptography

Classical RSA and Diffie-Hellman are secure only with large, costly keys. Elliptic curves deliver the same security far more cheaply, which is why they now dominate new systems and why they deserve their own treatment.

RSA and classical Diffie-Hellman are secure only with large keys (2048 bits and up), which cost
bandwidth, storage, and computation, an increasing burden on mobile and embedded devices. **Elliptic-curve
cryptography (ECC)** provides the same security with dramatically smaller keys by basing its hard problem
on the algebra of points on an elliptic curve rather than on integer factoring or modular exponentiation.

An elliptic curve over a finite field is the set of points satisfying an equation such as y^2 = x^3 + ax +
b, together with a special "point at infinity." One can define an addition operation on these points with
the same group structure that makes Diffie-Hellman work. The security rests on the **elliptic-curve
discrete logarithm problem**: given a base point G and a multiple Q = kG (adding G to itself k times),
recovering the scalar k is infeasible. Because no sub-exponential algorithm is known for this problem
(unlike factoring), ECC keys can be much shorter: a 256-bit elliptic-curve key offers security comparable
to a 3072-bit RSA key. This efficiency is why ECC now dominates new deployments.

Practitioners rarely choose curve parameters themselves; they use vetted *named curves*. Common choices
include the NIST curves P-256, P-384, and P-521, and the modern, rigidly designed **Curve25519** (used
for key exchange as X25519) and its signature counterpart **Ed25519**, both prized for speed,
simplicity, and resistance to implementation pitfalls. The elliptic-curve forms of the standard
protocols, ECDH/ECDHE for key exchange and ECDSA/EdDSA for signatures, are the defaults in TLS 1.3,
SSH, Signal, and most cryptocurrencies (Bitcoin and Ethereum use the curve secp256k1). When you need
asymmetric cryptography today, ECC is usually the right starting point unless interoperability forces
RSA.

A little geometry demystifies elliptic-curve "addition." Over the real numbers, to add two points P and Q
on the curve you draw the line through them, find the third point where that line intersects the curve,
and reflect it across the x-axis; to "double" a point you use the tangent line instead. Cryptography uses
the same rule over a finite field rather than the reals, which scrambles the geometry into something that
looks random while preserving the group structure that makes the math work. Scalar multiplication, adding
G to itself k times to get kG, is fast (using double-and-add), but inverting it to recover k from kG and G
is the hard elliptic-curve discrete-log problem. Because the best known attacks on well-chosen curves are
fully exponential, key sizes stay small: 256-bit curves for roughly 128-bit security. Practitioners must
still avoid pitfalls, using vetted curves (the rigidity of Curve25519 was designed to remove the suspicion
that surrounds some parameter choices), validating that received points actually lie on the curve to
prevent invalid-curve attacks, and using constant-time implementations to resist timing side channels.


## 2.13 Digital Signatures, Certificates, and PKI

Public-key encryption and key exchange both assume you know whose public key you hold. Signatures and the trust machinery of PKI are how that assumption is justified at internet scale.

A **digital signature** is the public-key analog of a handwritten signature, but far stronger. To sign,
the author hashes the message and transforms the hash with their *private* key; anyone can verify by
checking the signature against the message using the author's *public* key. A valid signature proves
three things at once: **integrity** (the message was not altered, or the hash would differ),
**authentication** (only the private-key holder could have produced it), and **non-repudiation** (the
signer cannot credibly deny it, since no one else has the private key). This last property, unique to
public-key methods, is why signatures underpin software distribution, legal documents, and blockchain
transactions. Standard signature algorithms are RSA-PSS, ECDSA, and EdDSA.

Signatures create a new problem, however: how do you know that a public key really belongs to the party
it claims to? If an attacker can substitute their own public key, they can impersonate anyone. The
answer is a **public-key infrastructure (PKI)**, a system of trust built on **digital certificates**. A
certificate is a data structure (in the X.509 format) that binds an identity (such as a domain name) to
a public key, and is itself *signed* by a trusted **certificate authority (CA)**. Your browser and
operating system ship with a list of trusted *root CAs*; a website presents a certificate signed by a CA
(possibly through a chain of intermediate CAs leading back to a trusted root), and your browser verifies
the chain of signatures up to a root it trusts. This delegated trust lets you authenticate a server you
have never met.

PKI is powerful but introduces its own risks and machinery: CAs can be compromised or coerced into
issuing fraudulent certificates, so mechanisms such as *certificate revocation* (CRLs and OCSP),
*Certificate Transparency* logs (public, append-only logs of issued certificates, often built on Merkle
trees), and short certificate lifetimes exist to detect and contain abuse. Certificate validation
failures, expired certificates, name mismatches, untrusted issuers, are among the most common security
warnings users see, and training users to click through them undermines the entire system.

It helps to know what a certificate actually contains and how validation really proceeds, since
certificate errors are among the most common security decisions an ordinary user faces. An X.509
certificate carries the *subject* (the identity, such as a domain name, possibly with Subject
Alternative Names for multiple hosts), the *subject public key*, the *issuer* (the CA that signed it), a
*validity period* (not-before and not-after dates), a *serial number*, key-usage constraints, and the
CA's *digital signature* over all of the above. To validate a server's certificate, a client performs
several checks in sequence: it confirms the certificate has not expired, that the requested hostname
matches the subject or a SAN, that the certificate has not been revoked (via CRL or OCSP), and, crucially,
that the signature chain leads from the server's certificate, through any intermediate CA certificates,
up to a *root* certificate already in the client's trust store. Each link is verified by checking that
the issuer's public key validates the next certificate's signature. If any check fails, the connection
should be refused. This chain of trust is powerful but only as strong as its weakest CA, which is why
incidents in which a CA was tricked or breached into issuing fraudulent certificates (for example the
2011 DigiNotar compromise, which led to that CA's removal from trust stores) are treated as serious
ecosystem-wide events, and why Certificate Transparency now requires CAs to log every certificate they
issue to public, append-only Merkle-tree logs that domain owners can monitor.


## 2.14 Putting It Together: The TLS Handshake

We have now assembled every primitive separately: key exchange, certificates, signatures, hashing, and authenticated encryption. TLS is where they all come together in the single most important protocol on the internet, so it makes an ideal capstone.

Transport Layer Security (TLS), the protocol behind the padlock in your browser and the "S" in HTTPS,
is the most important real-world application of everything in this chapter. It weaves together
asymmetric and symmetric cryptography, signatures, certificates, and key exchange to turn an insecure
network into a confidential, authenticated channel. The modern version, TLS 1.3 (2018), streamlined and
hardened the protocol, removing legacy ciphers and mandating forward secrecy.

At a high level, a TLS 1.3 handshake proceeds as follows. The client opens with a *ClientHello* that
offers its supported cipher suites and includes an ephemeral (EC)DH key share. The server replies with a
*ServerHello* containing its own key share, its **certificate**, and a **signature** over the handshake
that proves possession of the certificate's private key. Both sides now combine their ephemeral key
shares via (EC)DHE to derive a shared secret, run it through a KDF (HKDF) to produce symmetric session
keys, and switch to fast AEAD encryption (AES-GCM or ChaCha20-Poly1305) for all subsequent application
data. The ephemeral key exchange provides forward secrecy; the certificate and signature provide server
authentication and defeat the man-in-the-middle attack that plain DH would allow.

```{mermaid}
sequenceDiagram
    participant C as Client
    participant S as Server
    C->>S: ClientHello (cipher suites, ECDHE key share, random)
    S->>C: ServerHello (chosen suite, ECDHE key share, random)
    S->>C: Certificate (X.509, public key)
    S->>C: CertificateVerify (signature over handshake)
    S->>C: Finished (MAC over handshake)
    Note over C,S: Both derive shared secret via ECDHE, then HKDF -> session keys
    C->>S: Finished (MAC over handshake)
    Note over C,S: Encrypted application data (AES-GCM / ChaCha20-Poly1305)
```

Trace each arrow against the primitives in this chapter: key exchange (ECDHE), certificates and
signatures (PKI), key derivation (HKDF), and authenticated encryption (AEAD). TLS is a microcosm of
applied cryptography, and its history of attacks, BEAST, CRIME, POODLE, Heartbleed, and downgrade
attacks, is a catalog of how subtle the engineering is even when the underlying algorithms are sound.


## 2.15 Advanced and Emerging Cryptography

The cryptography above secures today's systems. The frontier, computing on encrypted data and surviving quantum computers, is moving quickly from theory into deployment, and a practitioner should understand where it is heading.

The cryptography above secures today's internet. A frontier of more powerful constructions is moving
from theory into practice, enabling computation on data that stays encrypted, and preparing for a future
in which quantum computers threaten the classical assumptions. This section surveys these topics at a
level that orients the reader; Chapter 17 develops their security implications further.

**Homomorphic encryption (HE)** allows computation directly on ciphertext, so that decrypting the result
yields the same answer as if the computation had been performed on the plaintext. *Partially* homomorphic
schemes support one operation indefinitely (textbook RSA is multiplicatively homomorphic, and the
Paillier cryptosystem is additively homomorphic, a property you can observe by adding ciphertexts).
*Fully homomorphic encryption (FHE)*, first realized by Craig Gentry in 2009, supports arbitrary
computation; it remains costly but is advancing rapidly and promises privacy-preserving cloud computing,
where a server processes your data without ever seeing it. **Functional encryption** generalizes
public-key encryption so that a specially issued key reveals only a specific *function* of the
plaintext (for example, only whether an encrypted record matches a query) and nothing more, with
attribute-based and identity-based encryption as important special cases.

**Oblivious computation** is a family of techniques for computing on data while hiding access patterns or
inputs. *Secure multi-party computation (MPC)* lets several parties jointly compute a function of their
private inputs while learning only the output, the classic illustration being two millionaires who learn
who is richer without revealing their wealth. *Oblivious transfer (OT)* is a primitive in which a receiver
obtains one of several items from a sender without the sender learning which, and it is a building block
for MPC. *Oblivious RAM (ORAM)* hides which memory locations a program accesses, defeating attacks that
infer secrets from access patterns, and *private information retrieval (PIR)* lets a user query a database
without revealing the query. *Zero-knowledge proofs*, in which one party proves a statement is true while
revealing nothing beyond its truth, are closely related and now central to privacy-preserving
blockchains.

**Steganography** differs in kind from all of the above: rather than making a message unreadable, it hides
the very *existence* of the message, for example by embedding data in the least-significant bits of an
image or audio file, where the change is imperceptible. Cryptography and steganography are complementary;
encrypting a message before hiding it provides both secrecy and concealment. Steganography is also a
defensive concern, since malware uses it for covert command-and-control and data exfiltration, a theme
revisited in the forensics and malware chapters. The small code cell demonstrates least-significant-bit
image steganography and additive homomorphism.

```{admonition} Going Deeper (graduate/research): the quantum threat and post-quantum cryptography
:class: tip
Large-scale quantum computers would break much of today's public-key cryptography. *Shor's algorithm*
solves integer factoring and discrete logarithms in polynomial time, which would defeat RSA, classical
Diffie-Hellman, and elliptic-curve cryptography entirely. *Grover's algorithm* gives only a quadratic
speedup against symmetric primitives, so doubling key and hash sizes (AES-256, SHA-384) restores their
security. The defensive response is **post-quantum cryptography (PQC)**: public-key algorithms based on
problems believed hard even for quantum computers, principally *lattice* problems, but also hash-based,
code-based, and multivariate constructions. In 2024 NIST standardized the first PQC algorithms,
including ML-KEM (the lattice-based key-encapsulation mechanism derived from CRYSTALS-Kyber), ML-DSA
(derived from CRYSTALS-Dilithium), and the hash-based signature SLH-DSA (derived from SPHINCS+). A
pressing operational concern is the *harvest-now, decrypt-later* attack, in which adversaries record
encrypted traffic today to decrypt once quantum computers mature, which makes migrating long-lived
secrets to PQC, often via *hybrid* classical-plus-PQC schemes, an urgent rather than hypothetical task.
```

Zero-knowledge proofs deserve a closer look, because they have moved from theoretical curiosity to
deployed technology. A **zero-knowledge proof** lets a *prover* convince a *verifier* that a statement is
true while revealing nothing beyond the fact of its truth, not the underlying secret. The textbook
intuition is the "Ali Baba cave": a prover repeatedly demonstrates they can open a secret door without
ever showing the verifier the password, by emerging from whichever side the verifier randomly demands.
Formally, a zero-knowledge proof must satisfy *completeness* (a true statement is accepted), *soundness*
(a false statement is rejected except with negligible probability), and the *zero-knowledge* property
(the verifier learns nothing they could not have produced alone). Modern *non-interactive* variants,
notably zk-SNARKs and zk-STARKs, compress such proofs to small sizes that anyone can verify, and they
now power privacy-preserving cryptocurrencies (proving a transaction is valid without revealing its
amounts or parties) and scalability systems that prove the correctness of large computations cheaply.
Zero-knowledge proofs combine naturally with the secure-computation primitives above, and together they
point toward a future in which one can prove properties of data, and compute on it, without ever exposing
it, which is why this area is among the most active in both research and industry.

### Computing Paradigms: Mainframes, Classical, DNA, and Quantum

Because the security of modern ciphers rests on what is *computationally* feasible, the nature of the
computer doing the work matters, and several computing paradigms, past and emerging, frame both
cryptography and cryptanalysis. **Mainframes**, the large, centralized computers that dominated
mid-twentieth-century computing and still run critical banking and government workloads, established the
multi-user, high-reliability batch processing on which early cryptographic and security models were
built; they are classical computers in the sense below, distinguished by scale and reliability rather
than by a different model of computation. **Classical computers**, from mainframes to laptops to
smartphones, all share the same foundation: they manipulate bits that are definitely 0 or 1 using
Boolean logic, and (per the Church-Turing model) they are equivalent in what they can compute, differing
only in speed and memory. The security of AES, RSA, and elliptic-curve cryptography is calibrated against
the best algorithms running on classical computers, which is why key sizes are chosen so that brute force
would take longer than the age of the universe on any foreseeable classical hardware.

Two non-classical paradigms change this calculus. **DNA computing** uses strands of DNA and biochemical
reactions to perform computation, exploiting the massive parallelism of manipulating trillions of
molecules at once; Leonard Adleman (the A in RSA) demonstrated in 1994 that DNA could solve a small
instance of a hard combinatorial problem. DNA computing is of research interest for certain massively
parallel search problems and for ultra-dense data storage, but it is slow, error-prone, and impractical
for general cryptanalysis, so it does not currently threaten modern ciphers. **Quantum computers**, by
contrast, use *qubits* that can exist in superpositions of 0 and 1 and become entangled, enabling certain
algorithms to explore many possibilities at once. As Section 2.15 details, this is not a generic speedup
but a targeted one: Shor's algorithm would efficiently solve the factoring and discrete-logarithm
problems that secure RSA and elliptic-curve cryptography, breaking them outright, while Grover's
algorithm offers only a quadratic speedup against symmetric ciphers and hashes, which larger keys offset.
This asymmetry, devastating to public-key cryptography but survivable for symmetric cryptography with
bigger keys, is precisely why the migration to post-quantum algorithms targets key exchange and
signatures first. The broad lesson is that cryptographic strength is always relative to the adversary's
computational model, and a paradigm shift in computing, as quantum computing may bring, forces a
corresponding shift in cryptography.

### Searchable, Deniable, and Functional Encryption

The advanced primitives above mostly preserve confidentiality outright; a further family relaxes or
reshapes confidentiality to enable specific functionality, and each is an active research area. **Searchable
encryption (SE)** lets a server *search* over ciphertext without decrypting it: a client stores encrypted
documents and later submits an encrypted query (a "trapdoor"), and the server returns the matching
ciphertexts while learning as little as possible about the data or the query. Symmetric searchable
encryption (SSE) is efficient and practical, public-key variants (PEKS, public-key encryption with keyword
search) allow others to write searchable encrypted data, and the central research tension is the trade-off
between efficiency and *leakage*: most practical schemes reveal some access and search patterns, which
leakage-abuse attacks can exploit, so the field studies exactly how much leakage is safe. Searchable
encryption underpins encrypted databases and privacy-preserving cloud storage.

**Deniable encryption** addresses coercion: a deniable scheme lets a single ciphertext decrypt to *two
different plausible plaintexts under two different keys*, so that a user compelled to reveal a key can hand
over a key that yields an innocuous message while the real message stays hidden under another key. The
ciphertext alone gives an adversary no way to prove that a second, "real" plaintext exists, which provides
protection against rubber-hose (coercive) attacks and is relevant to whistleblowers and censorship
resistance. Deniability can be sender-, receiver-, or bi-deniable, and it connects to plausible-deniability
features in some disk-encryption and messaging systems.

**Functional encryption (FE)**, introduced in the earlier advanced section, generalizes public-key
encryption so that a decryptor holding a function-specific key learns only *f(plaintext)*, not the
plaintext itself; attribute-based and identity-based encryption are important special cases, and *function
secret sharing* and *distributed point functions* are closely related tools (active topics in current
research, for example in private information retrieval and anonymous communication). Together, searchable,
deniable, and functional encryption illustrate a broad theme of modern cryptography: rather than treating
encrypted data as an opaque blob, these schemes allow precisely controlled computation, search, or
disclosure over ciphertext, expanding what is possible while carefully bounding what is leaked.

### The Algebra Beneath Cryptography: Finite Fields, Abelian and Non-Abelian Groups

Almost every cryptosystem in this chapter is built on *abstract algebra*, and a little of its vocabulary
clarifies why the schemes work and where new ones come from. A **group** is a set with an operation that is
associative, has an identity element, and gives every element an inverse. A group is **abelian
(commutative)** if the operation order does not matter (a * b = b * a) and **non-abelian** otherwise. Most
classical public-key cryptography lives in *abelian* groups: RSA works in the multiplicative group of
integers modulo n, and Diffie-Hellman and elliptic-curve cryptography work in cyclic abelian groups where
the discrete-logarithm problem is hard. A **finite field** (Galois field, written GF(p) or GF(2^n)) is a
finite set supporting addition, subtraction, multiplication, and division; AES, for instance, performs its
MixColumns and S-box arithmetic in GF(2^8), and finite fields underlie error-correcting codes, secret
sharing, and elliptic curves. Discrete mathematics, modular arithmetic, number theory, combinatorics, and
group theory, is therefore the true foundation of cryptography, which is why courses pair the two.

A frontier of research explores **non-abelian** structures for cryptography, where the operation does not
commute (as in certain matrix groups, braid groups, or group-ring constructions). The motivation is partly
the search for problems that resist quantum attacks: because Shor's algorithm efficiently solves the
*abelian* hidden-subgroup problem (which is why it breaks RSA and elliptic-curve cryptography), schemes
based on suitably hard *non-abelian* or non-commutative problems are studied as potential post-quantum
candidates, alongside the lattice-, code-, and hash-based families that NIST has standardized. Non-abelian
proposals remain largely research-stage and many early ones were broken, but they illustrate how the
algebraic structure a cryptosystem rests on directly determines both its security and its vulnerability to
particular attacks. For the reader, the practical takeaway is that understanding the underlying algebra,
which group, which field, which hard problem, is what lets one reason about a cryptosystem rather than
merely use it.

```{admonition} Going Deeper (graduate/research): why "abelian" matters for quantum security
:class: tip
Shor's algorithm is, at its core, an efficient quantum solution to the *hidden-subgroup problem (HSP)*
over *finite abelian groups*. Integer factoring and discrete logarithms reduce to abelian HSP instances,
which is exactly why RSA, Diffie-Hellman, and elliptic-curve cryptography fall to a large quantum computer.
For *non-abelian* groups, no general efficient quantum algorithm for the HSP is known; the symmetric and
dihedral hidden-subgroup problems, for instance, remain hard, and the dihedral HSP connects to the
lattice problems behind post-quantum schemes such as ML-KEM. This is the deep reason the post-quantum
migration moves toward lattice-, code-, hash-, and multivariate-based hardness, and why non-commutative
cryptography is studied: the *structure* of the group determines whether a quantum Fourier transform can
break it. This links the finite-field and group-theory foundations of this section directly to the quantum
threat and post-quantum cryptography discussed earlier in the chapter.
```


In [ ]:
# Chapter 2 -- LSB image steganography and additive homomorphism (self-contained)
import numpy as np

# --- Least-significant-bit steganography ---
def hide(cover, message):
    bits = "".join(f"{b:08b}" for b in message.encode()) + "00000000"  # null terminator
    flat = cover.flatten().copy()
    for i, bit in enumerate(bits):
        flat[i] = (flat[i] & 0xFE) | int(bit)     # overwrite the least significant bit
    return flat.reshape(cover.shape)

def reveal(stego):
    flat = stego.flatten()
    bits, out = "", []
    for v in flat:
        bits += str(v & 1)
        if len(bits) == 8:
            if bits == "00000000": break
            out.append(int(bits, 2)); bits = ""
    return bytes(out).decode(errors="ignore")

cover = np.random.randint(0, 256, (64, 64), dtype=np.uint8)
stego = hide(cover, "meet at noon")
print("Hidden message recovered:", repr(reveal(stego)))
print("Max pixel change:", int(np.abs(cover.astype(int) - stego.astype(int)).max()), "(imperceptible)\\n")

# --- Additive homomorphism (toy Paillier-style intuition with simple modular adds) ---
# Demonstrate that some encryptions let you add underneath encryption.
# Here: textbook RSA is *multiplicatively* homomorphic: E(a)*E(b) = E(a*b)
def egcd(a,b):
    return (a,1,0) if b==0 else (lambda g,x,y:(g,y,x-(a//b)*y))(*egcd(b,a%b))
n, e = 3233, 17    # toy RSA modulus (61*53)
Ea, Eb = pow(7,e,n), pow(3,e,n)
print("E(7)*E(3) mod n decrypts to 7*3 = 21 ->", (Ea*Eb) % n, "is E(21); homomorphic property holds")

## 2.16 Key Management

If the previous sections taught how algorithms work, this one addresses the problem that breaks more
real systems than any algorithmic weakness: managing the keys those algorithms depend on. A cipher is
only as strong as the secrecy and integrity of its keys, and the unglamorous discipline of *key
management* spans the entire lifecycle of a key, generation, distribution, storage, use, rotation,
revocation, and destruction.

**Generation** must use a CSPRNG with adequate entropy, as Section 2.4 stressed; a key guessable
because of weak randomness is no key at all. **Distribution** is the classic problem that public-key
cryptography and key-exchange protocols solve, but symmetric keys shared out of band must travel over an
already-secure channel. **Storage** is where keys most often leak: hard-coding keys in source code,
committing them to version control, or leaving them in configuration files and environment variables has
exposed countless organizations. The defenses are dedicated **key vaults** and, for the highest
assurance, **hardware security modules (HSMs)**, tamper-resistant devices that generate and use keys
internally so the private key never leaves the hardware in plaintext. On endpoints, a **Trusted Platform
Module (TPM)** plays an analogous role, anchoring disk-encryption and platform-integrity keys in
hardware.

**Use** should honor *key separation* (a key serves one purpose, for example never reusing an
encryption key as a signing key) and *cryptoperiods* (limits on how much data or time a key covers).
**Rotation** periodically replaces keys to limit the damage of an undetected compromise and to bound the
data encrypted under any single key. **Revocation** invalidates keys and certificates known or suspected
to be compromised, the PKI machinery of Section 2.13. Finally, **destruction** ensures retired keys
cannot be recovered, which is also the basis of *crypto-shredding*: deliberately destroying the key to
render encrypted data permanently unreadable, a powerful tool for data disposal and the right to
erasure. A related governance question is **key escrow**, storing a copy of keys with a trusted third
party so data can be recovered (or lawfully accessed); escrow improves recoverability but creates a
high-value target and civil-liberties concerns, a tension explored further in the privacy and law
chapter. Mature organizations formalize all of this in a key-management policy and increasingly automate
it through key-management services, because manual key handling does not scale and does not survive
employee turnover.

For the highest-value keys, organizations apply two further controls worth knowing by name. **Split
knowledge** divides a key (or the secret that protects it) so that no single person holds the whole
thing; reconstructing it requires several authorized people to combine their parts, which prevents any
lone insider from misusing it. The formal mechanism is **secret sharing**, most famously Shamir's
scheme, which splits a secret into n shares such that any threshold t of them reconstructs it but any
fewer reveal nothing, often described as M-of-N control. **Dual control** similarly requires two
authorized operators to act together for sensitive key operations, the cryptographic analog of the
two-person rule for launching weapons. These controls trade some convenience for strong protection
against insider compromise and accidental loss, and they appear directly in standards governing payment
systems and certificate authorities, connecting this section to the governance and compliance material
later in the book.


## 2.17 A Taxonomy of Cryptographic Attacks

Having seen how cryptography is built, we close the technical material by cataloguing how it is broken,
because choosing and configuring cryptography well means anticipating the adversary. CISSP's Domain 3
tests exactly this vocabulary, and the categories below recur throughout the book.

Attacks divide first by what the adversary can access. In a **ciphertext-only** attack the adversary
sees only ciphertext; in a **known-plaintext** attack they have some plaintext-ciphertext pairs; in a
**chosen-plaintext** attack (CPA) they can encrypt messages of their choosing; and in a
**chosen-ciphertext** attack (CCA) they can submit ciphertexts for decryption and observe the result.
Modern schemes are designed to resist the strongest of these, which is why IND-CPA and IND-CCA are the
benchmark security goals introduced earlier.

A second family targets the mathematics or the key. **Brute-force** attacks try every key and are
defeated only by sufficient key length. **Birthday attacks** exploit the square-root collision bound to
forge hashes or MACs with shorter digests. **Mathematical** attacks attempt to solve the underlying hard
problem more cleverly (improved factoring or discrete-log algorithms), and quantum algorithms, discussed
in Section 2.15, are the extreme case. **Frequency analysis**, from Section 2.2, is the classical
ancestor of all statistical attacks.

The most important family in practice attacks the *implementation* rather than the algorithm.
**Side-channel** attacks extract secrets from physical leakage, timing, power consumption, electromagnetic
emissions, or even sound, and timing attacks in particular have repeatedly recovered keys from naive
code, which is why constant-time implementations and constant-time comparisons matter. **Fault-injection**
attacks deliberately induce hardware errors to expose keys. **Padding-oracle** attacks (Section 2.8) turn
a decryptor's error behavior into a plaintext leak. **Replay** attacks resend captured valid messages,
defeated by nonces and timestamps. **Downgrade** attacks trick parties into negotiating weaker, breakable
parameters, the reason protocols must refuse obsolete ciphers. And the **man-in-the-middle** attack,
which plain Diffie-Hellman invites, is defeated only by authentication. The unifying insight, repeated
deliberately throughout this chapter, is that strong algorithms are necessary but never sufficient:
security is a property of the whole system, including its randomness, its key management, its protocol
design, and the code that implements it.

## 2.18 Applied Cryptographic Systems

The TLS handshake showed primitives combining into one protocol. To make the chapter's relevance
concrete, this section briefly surveys the other systems you use every day that are, underneath, the same
building blocks rearranged. Each is developed further in later chapters; the goal here is to see the
cryptography at work.

**Full-disk and file encryption** protect data at rest. Tools such as BitLocker, FileVault, and LUKS
encrypt entire volumes with a symmetric cipher (AES in a length-preserving mode such as XTS designed for
storage), while the disk key is itself protected by a key derived from the user's password and, ideally,
sealed in a TPM so that the disk cannot be decrypted on different hardware. This is why a stolen but
powered-off encrypted laptop usually protects its data, and why the *possession* property of the
Parkerian hexad, lost when the hardware is taken, still does not imply loss of *confidentiality*.

**Secure messaging** brought strong end-to-end encryption (E2EE) to billions of people. The Signal
protocol, used by Signal, WhatsApp, and others, combines an initial ECDH key agreement (X3DH) with the
**double ratchet**, which derives a fresh key for every message so that compromising one key exposes
neither past nor future messages, properties called forward secrecy and post-compromise security. E2EE
means the service provider itself cannot read message contents, which is also why it is a recurring point
of tension with law enforcement, revisited in the privacy and law chapter.

**Virtual private networks (VPNs)** create an encrypted tunnel across an untrusted network. IPsec and the
modern, compact WireGuard protocol authenticate the endpoints and then carry traffic under authenticated
encryption, applying the same key-exchange-then-symmetric pattern as TLS at the network layer. **Code
signing** uses digital signatures so that operating systems and app stores install only software whose
signature verifies against a trusted developer key, the defense whose subversion made the SolarWinds
supply-chain attack so damaging. **Cryptocurrencies** combine hash-linked Merkle-tree ledgers with
elliptic-curve signatures: ownership of funds is simply possession of a private key, which makes key
management (Section 2.16) a matter of real money and explains why lost or stolen keys translate directly
into lost or stolen value. Across all of these, notice the recurring grammar of applied cryptography:
agree on a key, encrypt with AEAD, authenticate with signatures or MACs, and manage keys carefully. Once
you see that grammar, unfamiliar secure systems become readable.

## 2.19 Practical Guidance: Choosing and Using Cryptography

After surveying primitives, attacks, key management, and applications, a practitioner needs a compact set
of rules for making real decisions, which this section provides as the chapter's actionable conclusion.

The first and most important rule is **do not roll your own cryptography**. Implementing ciphers,
protocols, or even padding from scratch is how subtle, fatal bugs enter systems; use well-maintained,
widely reviewed libraries (such as libsodium, the cloud providers' key-management services, or a
language's vetted standard library) and use their high-level "easy" interfaces, which make the safe
choice the default. The second rule is **prefer authenticated encryption** (AEAD) for confidentiality so
that integrity comes for free; never deploy a mode that provides confidentiality alone. The third is to
choose modern algorithms with adequate parameters, summarized below as sensible defaults for new systems
as of the mid-2020s.

| Need | Recommended default | Avoid |
|---|---|---|
| Symmetric encryption | AES-256-GCM or ChaCha20-Poly1305 (AEAD) | ECB; unauthenticated CBC; DES/3DES; RC4 |
| Hashing | SHA-256, SHA-3, or BLAKE2 | MD5, SHA-1 |
| Password storage | Argon2id (or scrypt/bcrypt) with unique salt | plain or fast-hashed passwords |
| Key exchange | X25519 / ECDHE (ephemeral, forward secret) | static DH; non-ephemeral exchange |
| Public-key encryption/signatures | RSA-3072+ with OAEP/PSS, or Ed25519/ECDSA | textbook RSA; short keys |
| Random values | OS CSPRNG (`secrets`, `getrandom`) | `random`, LCGs, time-seeded RNGs |
| Long-term/quantum exposure | hybrid classical + ML-KEM/ML-DSA | classical-only for decades-long secrets |

Two further disciplines distinguish mature programs. **Crypto-agility** is designing systems so
algorithms can be replaced without re-architecting, which the looming post-quantum migration makes
urgent; hard-coding a single cipher or key size is now considered a design flaw. **Compliance and
validation** matter in regulated environments: standards such as FIPS 140-3 certify cryptographic
modules, and frameworks discussed in later chapters often mandate specific algorithms and key lengths.
Finally, remember that cryptography is a means, not an end. It enforces confidentiality, integrity, and
authentication, but it cannot secure a compromised endpoint, fix a flawed protocol, or compensate for
poor key management. Used correctly, within a system designed with the principles of Chapter 1, it is the
strongest tool we have; used carelessly, it provides a false sense of security that can be worse than
none at all.

## Why This Matters

Cryptography is the load-bearing wall of digital security. Every other chapter leans on it: secure
networking is TLS, secure authentication stores password hashes and verifies signatures, blockchains are
hash chains and signatures, and privacy engineering increasingly relies on the advanced primitives in
Section 2.15. Just as important is knowing cryptography's failure modes, weak randomness, key and nonce
reuse, broken hashes, missing authentication, padding oracles, downgrade attacks, because in practice
systems are almost never broken by defeating AES or RSA head-on; they are broken at the seams, where the
mathematics meets fallible engineering. A practitioner who understands both the guarantees and their
preconditions can choose, configure, and audit cryptographic systems correctly, which is exactly what
certifications from Security+ through CISSP test, and what real adversaries probe.

With the mathematical foundations of security in place, the next chapter turns to the networks over which all of this cryptography travels: how data moves, how the protocols work, and how both are attacked and defended.


## News in Focus: Heartbleed (2014)

In April 2014, researchers disclosed **Heartbleed**, a vulnerability in the widely used OpenSSL library
(tracked as CVE-2014-0160). The flaw was not in any cryptographic algorithm but in a memory-handling bug
in OpenSSL's implementation of the TLS heartbeat extension: a missing bounds check let an attacker request
more data than they had supplied, causing the server to return up to 64 kilobytes of adjacent memory per
request. That memory could contain private keys, session cookies, usernames, and passwords, and the
attack left no trace in normal logs.

Through this chapter's lens, Heartbleed is a textbook example of the gap between sound algorithms and
fragile implementations. TLS, RSA, and the ciphers involved were all secure; the breach came from a
buffer over-read in C code, the kind of memory-safety failure examined in the exploitation chapter. The
response illustrates cryptographic operations under stress: affected organizations had to patch OpenSSL,
*revoke and reissue* TLS certificates because private keys might have leaked, and force password resets at
scale, exactly the PKI revocation machinery described in Section 2.13. The episode also accelerated
interest in memory-safe languages and in funding for critical open-source infrastructure. These details
reflect public reporting from the time and may be refined by later analysis.


## Review Questions

**Q1.** Kerckhoffs's principle states that a cryptosystem should be secure even if:
A. The key is short  B. Everything except the key is public  C. The algorithm is secret  D. No one attacks it

**Q2.** The one-time pad achieves perfect secrecy only if the key is:
A. 128 bits  B. Reused for efficiency  C. Truly random, secret, and as long as the message  D. A strong password

**Q3.** Why is ECB mode insecure for most data?
A. It is too slow  B. Identical plaintext blocks yield identical ciphertext blocks  C. It needs no key  D. It cannot decrypt

**Q4.** Which generator is safe for cryptographic keys?
A. A linear congruential generator  B. Python's default `random`  C. The Mersenne Twister  D. A CSPRNG such as `secrets`/os.urandom

**Q5.** The birthday paradox implies that finding a collision in an n-bit hash takes about:
A. 2^n work  B. 2^(n/2) work  C. n work  D. n^2 work

**Q6.** HMAC is preferred over `hash(key || message)` because the latter is vulnerable to:
A. Brute force  B. Length-extension attacks  C. Birthday attacks  D. Padding oracles

**Q7.** AES-GCM provides, beyond confidentiality:
A. Nothing extra  B. Compression  C. Integrity and authentication (AEAD)  D. Key exchange

**Q8.** RSA's security primarily rests on the hardness of:
A. Discrete logarithms  B. Factoring large integers  C. The elliptic-curve discrete log  D. Hashing

**Q9.** Forward secrecy ensures that:
A. Keys never expire  B. Past sessions stay secret even if the long-term key is later stolen
C. The future is encrypted  D. Certificates never need revocation

**Q10.** A 256-bit elliptic-curve key offers security roughly comparable to an RSA key of:
A. 256 bits  B. 512 bits  C. 1024 bits  D. 3072 bits

**Q11.** Which property is unique to digital signatures (versus a MAC)?
A. Integrity  B. Authentication  C. Non-repudiation  D. Confidentiality

**Q12.** For storing user passwords, the best practice is:
A. Plain SHA-256  B. Reversible encryption  C. A salted, memory-hard KDF such as Argon2id  D. Base64 encoding

**Q13.** In a TLS 1.3 handshake, the shared session keys are derived using:
A. ECB  B. (EC)DHE key exchange fed into HKDF  C. The server password  D. MD5

**Q14.** Shor's algorithm, on a large quantum computer, would break:
A. AES-256  B. SHA-3  C. RSA and elliptic-curve cryptography  D. Argon2

**Q15.** Steganography differs from encryption in that it:
A. Uses larger keys  B. Hides the existence of the message rather than its content
C. Is always unbreakable  D. Requires a certificate authority

**Q16.** Reusing a nonce with the same key in AES-GCM can:
A. Improve speed safely  B. Leak plaintext relationships and enable forgeries  C. Strengthen the key  D. Do nothing

---

### Answer Key

1: B 2: C 3: B 4: D 5: B 6: B 7: C 8: B 9: B 10: D 11: C 12: C 13: B 14: C 15: B 16: B

**Q17.** In ElGamal encryption, reusing the ephemeral value k for two messages is dangerous because it:
A. Speeds up decryption  B. Leaks the relationship between the two plaintexts  C. Changes the public key
D. Has no effect

**Q18.** ElGamal encryption is probabilistic because:
A. The key is random  B. A fresh random k makes each encryption of the same message differ  C. It uses a
hash  D. The prime changes

**Q17:** B  **Q18:** B


## Hands-on Labs

**Lab 2.1 (beginner) - Break a cipher.** Use the Section 2.2 code to brute-force the Caesar challenge
strings, then take a longer monoalphabetic ciphertext and break it with frequency analysis. Write a short
explanation of which language features made the break possible.

**Lab 2.2 (beginner/intermediate) - Randomness matters.** Generate 1000 values from the toy LCG and from
`secrets`. Plot histograms and successive-pair scatter plots of each. Identify visible structure in the
LCG output and explain why it disqualifies the LCG for key generation.

**Lab 2.3 (intermediate) - Modes and integrity.** Encrypt the same image in ECB and CBC using the
Section 2.6 code and compare. Then flip one byte of a CBC ciphertext, decrypt, and observe the result;
repeat with AES-GCM and show that decryption fails (tamper detected). Explain the difference.

**Lab 2.4 (intermediate) - Password cracking economics.** Hash a small list of weak passwords with plain
SHA-256 and time how many guesses per second you can check; then do the same with PBKDF2 at 200,000
iterations and with Argon2id. Estimate how key stretching changes an attacker's cost per cracked password.

**Lab 2.5 (advanced/research) - Build a mini-PKI and a homomorphic tally.** (a) Create a self-signed root
CA, issue a server certificate, and validate the chain with a library. (b) Implement an additively
homomorphic voting tally using the Paillier cryptosystem (via a library): encrypt individual votes, sum
the ciphertexts, and decrypt only the total, confirming individual votes are never revealed. Discuss the
threat model and what the scheme does and does not protect.

**Lab 2.6 (intermediate) - Inspect a real TLS handshake in Wireshark.** Capture traffic while visiting an
HTTPS site, then filter for `tls` (or `ssl`). Locate the ClientHello and ServerHello and answer: which TLS
version is negotiated (note the version constant, for example 0x0303 for TLS 1.2 or the TLS 1.3 supported-
versions extension); how many random bytes appear in each hello; the session ID length; and the chosen
cipher suite (name and value). Identify who sends the Certificate and who sends the Change Cipher Spec.
Then visit a site still offering an older configuration and compare the negotiated cipher suite, explaining
why a suite such as TLS_RSA_WITH_RC4_128_SHA is now considered weak. Map each observed message to the
handshake described in Section 2.14.


## References

1. Shannon, C. E. "Communication Theory of Secrecy Systems." *Bell System Technical Journal*, 28(4), 1949.
2. Diffie, W., and Hellman, M. "New Directions in Cryptography." *IEEE Transactions on Information
   Theory*, 22(6), 1976.
3. Rivest, R., Shamir, A., and Adleman, L. "A Method for Obtaining Digital Signatures and Public-Key
   Cryptosystems." *Communications of the ACM*, 21(2), 1978.
4. Nakov, S. *Practical Cryptography for Developers.* https://cryptobook.nakov.com
5. Boneh, D., and Shoup, V. *A Graduate Course in Applied Cryptography.* https://crypto.stanford.edu/~dabo/cryptobook/
6. Katz, J., and Lindell, Y. *Introduction to Modern Cryptography*, 3rd ed. CRC Press, 2020.
7. National Institute of Standards and Technology. *FIPS 197: Advanced Encryption Standard (AES)*, 2001.
8. National Institute of Standards and Technology. *FIPS 203 (ML-KEM), FIPS 204 (ML-DSA), FIPS 205
   (SLH-DSA): Post-Quantum Cryptography Standards*, 2024.
9. Gentry, C. "Fully Homomorphic Encryption Using Ideal Lattices." *STOC*, 2009.
10. Rescorla, E. *The Transport Layer Security (TLS) Protocol Version 1.3.* RFC 8446, IETF, 2018.
11. Vaudenay, S. "Security Flaws Induced by CBC Padding." *EUROCRYPT*, 2002.
11. ElGamal, T. "A Public Key Cryptosystem and a Signature Scheme Based on Discrete Logarithms." IEEE Transactions on Information Theory, 1985.
12. Crypto++ Library. ElGamal encryption reference implementation (accompanying source: cryptopp-elgamal.cpp).

**Related work by the author (see Appendix E):**

- Trivedi, D. (2022). *Ciphers and Cryptanalysis*; (2025) *Cryptanalysis of the Vigenere Cipher.* (see Appendix E)
- Trivedi, D. (2021). *Message Integrity*; (2014) *Security of Cryptographic Hash Functions.* (see Appendix E)
- Trivedi, D. (2021). *Eavesdropping Attack, Cryptanalysis, and Pseudorandomness*; (2021) *Quantum Cryptography 101.* (see Appendix E)
- Trivedi, D. (2023). *The Future of Cryptography: Performing Computations on Encrypted Data*, ISACA Journal; (2026) *Which Cipher Is More Secure?* (see Appendix E)
- Trivedi, D., Boudguiga, A., Kaaniche, N., Triandopoulos, N. (2023). *SigML++*, Cryptography 7(4); Trivedi, D. (2011) *DNA Computing.* (see Appendix E)
- Companion code: *Python-FHEz* (Python 3 + Microsoft SEAL 4 toolkit for fully homomorphic encryption) and *chiku* (polynomial approximation for encrypted computation). See Appendix F.
13. Hahn, F., Peter, A., et al. Research on searchable/homomorphic encryption, function secret sharing, and privacy-preserving federated learning (University of Twente). https://research.utwente.nl/en/persons/florian-werner-hahn/
14. Boneh, D., Sahai, A., and Waters, B. "Functional Encryption: Definitions and Challenges." TCC, 2011.
15. Canetti, R., Dwork, C., Naor, M., and Ostrovsky, R. "Deniable Encryption." CRYPTO, 1997.
16. Curtmola, R., Garay, J., Kamara, S., and Ostrovsky, R. "Searchable Symmetric Encryption." CCS, 2006.


```{index} Plaintext / ciphertext, Cipher, Key, Symmetric cryptography, Asymmetric, Kerckhoffs's principle, Cryptanalysis, Hash function, MAC, AEAD, KDF, PKI
```
